In [ ]:
#@title Lesson 4 of 26: Classification, Losses and Features
from IPython.display import HTML, display
display(HTML('<div style="background:linear-gradient(118deg, #0d0726 0%, #2a0b5e 38%, #8f0f6b 72%, #ff6a1a 100%);border-radius:22px;padding:38px 36px 30px;color:#fff;font-family:\'Avenir Next\', \'Segoe UI\', \'Helvetica Neue\', Arial, sans-serif;box-shadow:0 12px 40px rgba(60,10,90,.35)"><div style="font-size:12px;letter-spacing:.34em;font-weight:700;color:#ffd1ea">BASIC PRINCIPLES OF ARTIFICIAL INTELLIGENCE  ·  LESSON 4 OF 26</div><div style="font-size:42px;font-weight:800;line-height:1.08;margin:14px 0 14px;text-shadow:0 2px 18px rgba(255,80,160,.45)">Classification, Losses and Features</div><div style="font-size:16px;line-height:1.55;max-width:680px;color:#f5ecff">How to train a yes-or-no decision: score and margin, the zero-one, hinge and logistic losses, the update that fixes a mistake, and word and bigram features for text.</div><div style="margin-top:22px"><a href="https://youtu.be/771lIGWNeNk" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:#fff;color:#4a0d66">▶ Watch the lesson</a><a href="https://www.youtube.com/playlist?list=PLDyCOTShSfCk" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:rgba(255,255,255,.16);color:#fff;border:1px solid rgba(255,255,255,.45)">All lessons</a><a href="https://colab.research.google.com/github/neural-modules/ai-principles-notebooks/blob/main/04_classification_losses_and_features.ipynb" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:#ffb22e;color:#1a0d00">Open in Colab</a></div></div><div style="background:#141033;color:#ece8ff;border-radius:18px;padding:22px 26px;font-family:\'Avenir Next\', \'Segoe UI\', \'Helvetica Neue\', Arial, sans-serif;font-size:15px;line-height:1.5;border:1px solid #2c2466"><div style="font-size:12px;letter-spacing:.3em;font-weight:700;color:#a87bff;margin-bottom:6px">HOW THIS NOTEBOOK WORKS</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#ff3d9a">1</span>Choose <b>File &gt; Save a copy in Drive</b> first, so your work is saved to your own account.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#a87bff">2</span>Run the set-up cell, then work down the page in order. Later exercises reuse earlier answers.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#2ad4ff">3</span>Replace each <code>...</code> with your answer and run the cell. The check runs straight away.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#ffb22e">4</span>Stuck? Open the 💡 hint under the exercise. Some exercises have two.</div><div style="margin-top:14px;font-size:14px"><span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#11c48a;color:#03251a;padding:3px 10px;border-radius:999px;font-weight:700">✅ correct</span> <span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#ff4d6d;color:#fff;padding:3px 10px;border-radius:999px;font-weight:700">❌ not yet, with a pointer</span> <span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#ffb22e;color:#2a1700;padding:3px 10px;border-radius:999px;font-weight:700">⏳ still blank</span></div><div style="margin-top:16px;font-size:12px;color:#9d95c9">© 2026 Applied Data Science (<a href="https://www.youtube.com/@neuralmodules" style="color:#ff8fc4">youtube.com/@neuralmodules</a>). All rights reserved. This notebook and the lesson video belong to the channel. They are for personal, non-commercial study. You may run and change your own copy. You may not republish, share, sell or reproduce this notebook, the videos or any part of them without written permission from the channel.</div></div>'))

In [ ]:
#@title Set-up: run this cell first
import base64, contextlib, html as _html, math, sys, zlib
import numpy as np

try:
    import torch
except ImportError:
    torch = None
try:
    import einops
except ImportError:
    get_ipython().run_line_magic("pip", "install -q einops")
    import einops


from IPython.display import HTML, display
import matplotlib.pyplot as plt
from cycler import cycler

PALETTE = {"pink": "#ff3d9a", "cyan": "#2ad4ff", "amber": "#ffb22e", "violet": "#a87bff",
           "mint": "#3dffb4", "orange": "#ff7a3d", "ink": "#ece8ff", "panel": "#141033", "night": "#0c0922"}
plt.rcParams.update({
    "figure.figsize": (6.8, 3.8), "figure.facecolor": "#0c0922", "axes.facecolor": "#141033",
    "savefig.facecolor": "#0c0922", "axes.edgecolor": "#3a3170", "axes.labelcolor": "#ece8ff",
    "axes.titlecolor": "#ffffff", "axes.titleweight": "bold", "axes.titlesize": 12,
    "text.color": "#ece8ff", "xtick.color": "#b9b2e6", "ytick.color": "#b9b2e6",
    "axes.grid": True, "grid.color": "#ffffff", "grid.alpha": 0.08, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False, "lines.linewidth": 2.4, "image.cmap": "plasma",
    "axes.prop_cycle": cycler(color=["#2ad4ff", "#ff3d9a", "#ffb22e", "#a87bff", "#3dffb4", "#ff7a3d"]),
})

_CARD = {"pass": ("linear-gradient(90deg,#0fae7a,#3dffb4)", "#03251a"),
         "wrong": ("linear-gradient(90deg,#e0245e,#ff6b6b)", "#ffffff"),
         "todo": ("linear-gradient(90deg,#ff9f1c,#ffd166)", "#2a1700")}

def _say(state: str, text: str):
    bg, fg = _CARD[state]
    try:
        display(HTML(f'<div style="background:{bg};color:{fg};padding:10px 16px;border-radius:12px;'
                     f'font-weight:600;font-family:sans-serif;margin:4px 0;max-width:860px">{text}</div>'))
    except Exception:
        print(text)


if sys.version_info < (3, 12):
    print("This notebook uses Python 3.12 syntax. Colab runs 3.12 or newer, so use Colab if a cell fails.")

class NotDone(Exception):
    """Raised when an answer is still the ... placeholder."""

_results: dict[str, str] = {}

def _ready(x):
    if x is ... or x is None:
        raise NotDone
    if isinstance(x, (tuple, list)):
        for v in x:
            _ready(v)
    return x

def _array(x):
    if torch is not None and isinstance(x, torch.Tensor):
        return x.detach().cpu().double().numpy()
    return np.asarray(x, dtype=float)

def close(answer, expected, tol: float = 1e-6, rel: float = 1e-6) -> bool:
    """True when answer matches expected to within tol (absolute) or rel (relative)."""
    _ready(answer)
    a, b = _array(answer), _array(expected)
    return a.shape == b.shape and bool(np.allclose(a, b, atol=tol, rtol=rel))

def same(answer, expected) -> bool:
    """Exact match, for counts, strings and choices."""
    _ready(answer)
    if isinstance(answer, str) and isinstance(expected, str):
        return answer.strip().lower() == expected.strip().lower()
    return answer == expected

@contextlib.contextmanager
def checking(label: str, praise: str = ""):
    try:
        yield
    except NotDone:
        _results[label] = "todo"
        _say("todo", f"⏳ <b>Exercise {label}</b> · replace the ... with your answer and run the cell again.")
    except NameError as e:
        _results[label] = "todo"
        _say("todo", f"⏳ <b>Exercise {label}</b> · run the earlier cells first ({_html.escape(str(e))}).")
    except AssertionError as e:
        _results[label] = "wrong"
        _say("wrong", f"❌ <b>Exercise {label}</b> · {_html.escape(str(e)) or 'not quite yet. Open the hint above and try again.'}")
    except (TypeError, AttributeError) as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            _results[label] = "todo"
            _say("todo", f"⏳ <b>Exercise {label}</b> · your code still returns ... or None. Finish it and run the cell again.")
        else:
            _results[label] = "wrong"
            _say("wrong", f"❌ <b>Exercise {label}</b> · your code raised {type(e).__name__}: {_html.escape(str(e))}")
    except Exception as e:
        _results[label] = "wrong"
        _say("wrong", f"❌ <b>Exercise {label}</b> · your code raised {type(e).__name__}: {_html.escape(str(e))}")
    else:
        _results[label] = "pass"
        _say("pass", f"✅ <b>Exercise {label}</b> · correct." + (f" {_html.escape(praise)}" if praise else ""))

_CHECKS: dict = {}

def check(label: str):
    """Run the check for one exercise by hand, for example check("3.6")."""
    fn, praise = _CHECKS[label]
    with checking(label, praise):
        fn()

def _auto_check(result):
    import re
    m = re.match(r"\s*# Exercise (\d+\.\d+)\b", result.info.raw_cell or "")
    if not m or m[1] not in _CHECKS:
        return
    if result.error_in_exec is not None:
        _results[m[1]] = "wrong"
        _say("wrong", f"❌ <b>Exercise {m[1]}</b> · the cell raised an error (see above). Fix it and run the cell again.")
        return
    check(m[1])

_ip = get_ipython()
for _f in list(_ip.events.callbacks["post_run_cell"]):
    if getattr(_f, "__name__", "") == "_auto_check":
        _ip.events.unregister("post_run_cell", _f)
_ip.events.register("post_run_cell", _auto_check)

def progress():
    """One bar per part: how many of its exercises pass so far."""
    parts: dict[str, list[str]] = {}
    for label, state in _results.items():
        parts.setdefault(label.split(".")[0], []).append(state)
    if not parts:
        return _say("todo", "No exercises checked yet. Run an exercise cell to start.")
    rows = []
    for s, states in sorted(parts.items(), key=lambda kv: int(kv[0])):
        done, n = states.count("pass"), len(states)
        rows.append(f'<div style="margin:5px 0;font-family:sans-serif;color:#ece8ff"><span style="display:inline-block;width:64px">Part {s}</span>'
                    f'<span style="display:inline-block;width:320px;height:12px;background:#2a2357;border-radius:6px;vertical-align:middle">'
                    f'<span style="display:block;width:{320 * done // n}px;height:12px;border-radius:6px;'
                    f'background:linear-gradient(90deg,#ff3d9a,#ffb22e)"></span></span> &nbsp;{done} of {n} checked</div>')
    display(HTML('<div style="background:#141033;padding:14px 18px;border-radius:14px">' + "".join(rows) + "</div>"))

_say("pass", f"✅ Ready. Python {sys.version.split()[0]} · NumPy {np.__version__} · "
            f"PyTorch {torch.__version__ if torch is not None else 'missing'}")

# The answer checks. Each runs by itself after you run its exercise cell.
exec(zlib.decompress(base64.b64decode("""
eNqtfW2X40SS7nd+hcZf2gaXx/JLld1s7UwDDdNnmYYLzQVOTV0f2ZZd2pYtI8ntMpz+Xff7/WU3nojMVGY6VQ3M7g5dtqV8i4yM
94hc/BhHt9FvnVWZVA/puvM8igejftQp081xj6/DwZy+Jtsk2/O3KX2rH5L924q+Xg0HN/T9l2Na1VmBF67iwfD9R4s3Y+r1rttJ
DodIdR1xH5Fq24/iXj/qUtNs9TbSHTRPr+SxTMM852b3Hy1+5u5jvEa/0A9vvnvx6jV++yii/+t28mJLY+2LOjoV5dtsvzUDHpLz
Lt3X/Gxb0BMasyyO2wfzBub8Nk0PlczcNFZdV1mdRlkVrYvT3rRZJeU6Wj0k5ZYWWp+yVWoeYaAj/U5j/vexqqPEXs1V02/6Li3P
NYZD72raNM3TBVAOZbav09K84wHW6vOQp0mV0lR5uau8oC+7c5SsVsVxXzc9PtAo9FKGJey3/E66S7Lc7e2UbTIbplFCq8lzs9Lj
IS+StTf7ZhC0TKJDWSzzdKdgUam94VFoHz//5usvBHPqhzTCTlBvoW20HjePrhg5vv3h9edvqJPOoP+Xv336/F+dZ91e56PFjzNG
dA+xqW/+HDPOr9LsXSoPJ/SDfygcvCcs//7ltzzZ7rgfjXpqat0RzsjYfB3z10HznIYaDmY9DZfucHBNj+xfYvxCU+jpJX336vVX
C0Fwen2Mh+ive8XzpF/whRdBv0zVkxF/G5n23/zwhjtQvUsHMWZHM5AOJpjItWo/xejSGy+fvsTyBYeOO/7oo3W6iRZ18bZbp491
7zljSpnWx3If3dWDqi6zQ1d2pBdtijKqI6AqvTvIi1NadnuD6pBndbcXZZvIa2D6X2bbskuDpPvKGyKJPok69P+fREvuPunTBxrh
V+pFGvQj+XsXP79vejw8ZIvjPrNnvQF6vJePeqL+yvjhXX1Pr24G27Tu1gS8Ho0e27Pa2KMsLwZxBzcD8rxlqc2wzrjLZtzlk+Ou
i7p76kcbF1gbOp11tzruuifu4y02nnr5OHrHE6Dv7zCHzYAo3K7q9no93eGvRXfndhZjv3bRf9xGwyjNiawM9bugYKn/+i55JPQa
Am+G0VW0Mz2DUBMxXPkNZLL7A6HJNlmv08eDtL/aNbOqsu2uyNbdX72p0RB/BZoOCUC7pH4YoPXVr03D42HN8OlH536U1olqv09P
BOB1tiLoyb7QGs8Enwag0X9EsbUlIag1j1Wfd2+xb/TBgfonGJj6Rv/v7OnTi+ZclUTYF+ukTrr4h+dK/xyK1YM+CDSrBwIg424/
urs36LTApEpQ867TQD/H4vEKOnanDCBcgsi8sqtARTyw+D3em9cxuwFR6nS/7jLyCTJxix1e31VEVPiBxhznGW1lnu679KnXs8Gk
Fk6gWnz7j1csDzQHS9GavkCBpBGigT8ufnj9CkuzoHrXtdrQ9E27s7SDVHHPxJII/LB3N+R+PmvpZvkHevny5Ys331M3VVHW6br7
21tvxizPMH7xd9P9+95Hi59u5kCowyApy+TcvbtrHjOKbZg48JZwYx7rPjAATWpdnw/pLZ826vnneOz0fBZU8lfiNFK4esrqh8Uv
3dMv7mHk07T4kTiGlnpu6RXdKC3LQBuNJQbHrM6BbS66/coMg1CAlvbzuDnkjE1tvQuq/bkBCCPHFvlqG8JQtn9/FFnKtkzWC6Jn
3Z/4UJ7UmDvaMQzR/Sn6e3RyzshV96fBG/qVp9DdgXj1evpEnRtIVdv1Yn/oftuPflY0huj1QsjGbTwcDjWxEeT4NS2LqvvtoHpI
DuldfN/wsPTQUJ3Y7gWsyiJAu4LELOrsy4RYh0OWHmltBh6YkEdPQZF5NY+8Wpce2xTs1FBYvP948ZKew5vy2EyBuoe0yc/cfhuy
kx48OtQsUwF09ZCu3i7iRay5QVJVaVlHVbJLSSxZ5MkyzUleZq2gowRD9RUS7Sdx54lmpK3U2T7taHlR/4CWV9Ry8fk/Xn7+X9/f
PSNB7RlYT7eZUD969szsu/515E2T9YRutSrKVI4u4yWRrR7z714jGUMMjA75sVI6CAmN8lUkZZaTP2U5/SGBHB+d0mz7UHfaRvut
k5B0PSIVsh/JZzoOnSUE8Pc07LUMnkSsvkTFJhpHu2NeZyRCphg0VQNEy3M0bh/F9M69DlWve9J5DvVZd1EnpKdEW1IIaHURt8SI
QwfAIw/AoyCAx+0AXmyysgJVZgLhALvHGge0txpa0/q44iWTwp6QSqzecmYz9mZDwKNfh88bTYabKm0+Ohyrh+h4ALBi1uOVfkMP
8lwUXDxjHV/aHQ7Uw3rNKhmI0uByrZMQzh/KlBmBYE/c4+6cJ6Rc3BiEPhSkZBPggdFyKvokFm0T/Zs+Aa3jyJZKZ9bepY/Jqs5p
RVqrTJZ00L0tnXhAnAS3dBrc0l1SbrN9N9bnBP/yWp3HV2y44PXKvzRHfYaJYbxNSd0lTkFYZxYtS8AhIg1aQ0e66zw1DTqAAIL6
AxxXVOZ3jGN+1ONYQJp6QJoGgXT9BJAqQs5GimTM3zzJEUF9OkwCafI7OpQ8175eCSGqfBJ4bLItIe0jTfKQk1p5XB7yoq669HOV
/ZredgmlR6RgC+tal8lpIdNaJI9Z1U0e+5E1TXN8bFtXYyQjiajzc3EsdZOO9MoDPxSnbs+G3LUHuWucUUYERhTs1CB6cyqIxpVl
Skc+2VekKFdyBAtCkX2a0FBZVYkNJU83TBfAlQPn8SZ0HnOSTLrURZ28TSsYD+6YE/cj9Qc8keF9eshWD3pdUUK4kdBgpKympL5H
w785SHHjLY1g/OybPZ03UDFlyZM9GkSvarWBOIpimKqLiBS8qqDX6bdMUFAtNbCwWRi5FppgXImRRaHMJ7HCGvwcOjILdQK1cUa1
u3LbWaudeaud0Wrf0EIB4GZpxKtytQgAmxEcmyi9Y0vBzfabbM2TdjddRHbqk6htWgZgMA/CQEC92OTZgcBAR+uuwVreVbUXitEx
wwYBJlq1A0alvxyT3KOKc2+1c1rti/05crsSegrNe5M9MkcmEOgtfxFVuyTPtY2RthuAUC0x2Q/teDzsGjl7o5XQrkM/PmHD1pNk
pJUgkQaV0DGjvgGjNCHEl20ss4rWwpxw9Cn2Y6+ljjOTnQe1na7oNfRlryFz4xHO+pR3Xo46tcbAUfWQbehQsBlYDSy8mfVGOYgP
aX6oXBJecVcPx7I2fBFwVfTi4uRFK5LdsRzaqEuMGoVF1jvQlgX16ijn3VE/0jZHgisREVohTDz3wtJOZbHfRlW23RNaVzS9gFRr
Oh4qyYBb1hnTmaHIebRCYIY6QRaQR558OwrKt6ML+bYhgb8Wi3dE8UAAfbNEV1lGh/IvrXJMU+SlxfKWJoo2QXRmN/JmBxMyOlTg
UDKUbNTV2PxK2gftPWQx3j4NDEEVQliil8n+GS0jJU5Ngwr1pDngy4aOT6L3OqsGnQtohIVR0sSLko5IDL0Ti8TZ1ESL0Aocpn4o
Uy2hVAEamqxWxzJZnWEN193E1Mn+KLI5j+FAaOxBaMwcA4vPyrUgEDH4UyE/EIKDVgSwdhJcE82HiDIprLs02Xcbq4byG+n/mv/d
wxjavDdSyC0mdpbZZgoZYm1dx/c5tfvPaGgEdgwHeHUvpBV+zYHAxIMApM0pGs9kvy+OAkR3otE8TrFO8z5OB77hKF0xawGKVMm5
CgBqGjoKvC/VIqmFCehjWDPzuyAg8LhgYy4P813T0Ts5SWwZVb4Ehht/wiAgF8r88+F3WeNklU/LLIAJU2itUFognXognQaJwnXb
MVgQEr9Lw9P78X8tvvru1Rf3+oDIpA7wxQkbK+TMbsts/fvk0Gtg11jJocnjoKrTQ1cN04+s6ZxIAkhvoRrBb7cq8qK8/fbF1y/f
vHl51zlk+7cQQ/PT7ajpKq0Xj8pmoKe3iZ5pUeBZp/epfu2sXtPgfUKCHXkS7Igl2DdMGjY5bUtVl2lNz0g8LYgZrZlF0hH+7+Pu
UA2ir8pknYF9rdNqhb9ExogbaTGRuzgkxM0C2HsT3DPuGC4rlnQamVy+K51W1Gk6qyQinH6J4LYA2aWPV1Bw8YODRDfeIiHLfpbW
p5SY/xW6EQZ+AweoSxhZShZCxSdYcESEHrXQBOeXRV6Cy4GgFFxtWMCt8uKQLrhTY7qozTDrIq3AG5SMtdQzjrVyMZ87y5x5y4QQ
qzjN8bBOalrJMSchrTjm60boYbFOST5A2Ai4zCPAcNYwNkIdiDNEkeDUrmEYDKxzHqJJq2K3zPYJELWCexW+0BnTJWKUD0W2Iliy
xRFEgNB6ok5g1Ql2VVSLU0EchAjKMPr442g8Z+Y0jIj9HyEsjedNBxZ85h58IPZSq7rgtR/gwqSmg+h7EltXDxJNkO4YKQhEhm+m
xvqtJTk8BYdl+TJNKhAO6hQ+BRCRRKi4Us95zy8gN74Q1gRD7i79KCKqTcWdDEHN0NfRYHjv82mRbGByweYZyeZiIBlHWIbCRGkK
72BlrC2KJm7ywuH+Y096Gwelt3GLdZLHhm5+YcoXTeDDpgRux1ahxKi4IaHm3XahBzM+BeaTPrtnYQ2UwFnkyFskLIQAPh8XdqUn
BOWEpkLK0xFy3viGdCVCIOqSzgxhL1C9LHaiCTZiWQAhwrId7exim5AuSB8sL9tV5Mu8/1y8+OnV92ZhrK4R2urFUR+B40WzJmWz
eCtWA91WoEvMA4RDI5LCJU02HDiNPTiNtTqtJ0GjA6uGz7k9c0VC1+JIm0f0VCHeUKTpZYFzQ0drKJAzaFnsB9HXOLXqqKp58mHj
gATW2NMsx3OoSHIgFFsMQDxo7pQTuOBj657Dq5E5eWMlBQGXWLbkB3IaiV5rs47Q9nKflkCaUrPxrL7cCXtU51Qe4Z8XrqM7c2A/
8WA/CR7EsMlzf1zToDy0XkVd5LdxenXNUiQjteOWGHsS2pglNEjihBTJ+jyI/qH4hmiQljrTB2HM6jpXlFPta5KfSNzVG5gRD7a2
VS2bzTIlcbTAJl63kxhLKr48/Y2IrM8wLD7mGLeRzEbC1D0+JWPqriFlQq50fCh/VLzE40a8dGbjiZTJbklY8j8gU+r582BPCJZj
T7Acs2D5Ahw6T88OK4RyXCvBSbw+yVmEFpZGSABccTThyqI+GI7bBfY/LFcyw4J9wN56GNIv6CMLEZrPMTg7oe6SesGuUQmv4igu
SDQ4LnRQ41j3yy4hpmN5AsLXbHrVzkxvPODBn/NC3sMWaayMx0IRMQG2uMIG9aJdhpUBG7G+MSuJmllnLOpA5hxeGBvGbUZaMbVX
x50vpZN07kjxAEkXYvYVieow73W1CN9zVj/zVj+DSDO4HkSfgRNoU43YTUpm+DE0KgbmGS4t2NoIHBwbV4llUrN0ktEVgTlAqY5e
7Nkcq1U+kY0fCM3RgQjeAUg0Ui5C1G6jRQla110m68UhycqeDSXw5S5C25Zw1o9gXeCAiAQQwIcle/FHQoKEQ635OFgiubDeKtNW
WcbNBxKC2xjHAT9VITYuT5RpErN1YD/3YD/XnBvTUI62VbF/lz42yCP9gqALOxJLeJ5fHM5Ji5BroiuExXGYF/3WHWnCrAUCyFBZ
pTB2i00ddZ7obuR2FzshZNQ5myIwM4E0jJsKT+gNwpq9bYuYeCLuJCjiTlpEXMSWwDoJRmGFydnSRCwG1z5QQYm1wFyhR2AYF667
iSeQTlggfcHUIR6MOTb1es4GL/5M9EI03eEUFnzPP2GdC8OZiWOklQI44qFLZsgykcD2tvjGVWjfUIkxU8+TO1SOeaz2Ick3nfYu
ZEdjDgkM7aZ5kv4fDhJ8oquxBO7pr2wRNiERTTSieLf1S/je0BdnK8beVoyD2BE2bh5Y+dGjBFQfjtiCznNvzc7oOwxJZzITbzIT
gxdEf4VZabfF5GYq7PbCMmg8efBqISqSfVA5aUgQqJV/IoAF05YTUFULsKsFHFnmUJpVCzdmvdMor+qQa35HU+20dLzOgMp9KwbV
4e76VyNFKoqClwJdHrQDcVET/ew30alw8fsaNoLgkyVpGfUZk1WBQPZuTL3dQNrFG8WNcvHgEDyYH+s9mkwsn48TTvCMRqVJKafj
lhm8ajWbWhYJRdI0ojTnFu7pC542aeRmJknvHA1nyLaFfjQZDK0Qrv1xx8tq6Nk7hIZBWYA+av98JT8zo6MPposg3da61juC9B5i
haWDbCx28Nvu3XtZJsuRikkutfL9G7V9Pphs3jt7ce3tBeRS2KnZMK83AGgxwi9ET5N8V1S1qFwTOSkca7NJ1qnyZumohVT0+yrZ
pPm5xb8xaRFQm8VT6+7VbCjUUv/tcEB8xExbNCk1qJHejLbi2346HxpMjzG09L145A0qAmTroMMPjqL0LoMVWu8ilq52LtlCamXd
gMXiotwd80S87OU62yc0DR2VYW/pjbelMO/SmiSHQCMLHQEAtTlVrEOKxRC7uEw3YEcIf2fr/CD6OluWGJIXqtUQRFKJyRS9n4tj
YINbbL2/HEkQXy92XeE0EzecqHnsaPvEzBBhH6nHIX72C0mfBLmqYovkaDCauv0utAmQH4F9MOMVNGEq0tfd9yVcoIpkWG5njW3B
fObBHLbmz7ShBgMNoh+xkdx/Beh/oszWxNYY/NKpQHedIqRcGdSVVtQqZARNzKSkduFzJRV/AV/lbx3tQuto8yA+aOzDZ72s99qq
Ikh3LKN9shOkeJueQ77ROzMS0gFMYHf3949pedibE4F5T+5NvILHqsYO/Oce/JH+JP2xUxqQJmJlAnYcaEtqWxU9pMm7LJew8GXK
PB2H3OIhp4eziNxQtQhEa353RZpslW2yFVv0A6ykie24kIGNtUSCrdtsJdrOcMG3GTIXlhNnmNMv3PkyrSRwKA5o5VYPDcGByQm6
uoz9P2GTedoo9ISZBqfmtmNbWS76tqDpdbM6J3urF4N/f9Twk6dbJHpsin3Ni52123smfojMhENkOggTKkAWbMMPRBEFa9Fzk1rt
U7bXCvA6Nzo7Q4GkHvYmsg0khHJhzXK5YlO/INulod9FL9vs30RG2Gu80ABjS66eaMoGxkKyB+2liAu8A9SVXh7IC9NJfhvE2LAf
S2gLac9PBHcvhnNL1YVh5a+RloCR5OP+6i7L1yLjRo0kUs7BXiuZHU31oQCr5lg3W/K1nul1sJH2CCUS5hGWdk+hSIZpOFCpSZTQ
Qd6HA8K8nVzO0XuJcNBPr9zHV6P3lidr+NzmeiqWY5Mm9ZHm3nlyAk76qES0ByZhDzZ6jvCfrfKQdz68PJOQGr9X4c7Ob6HQYjoV
egCO82Wn0oFjisWeYe3z1LNfTIP2i2ljv1AoeWtgOxyMJKVyW9SN0UvWIQ7mrrRp2ywsLBYLQC/AValbTnnitrRc8133oNydN9yF
8In1mvMUZCtNXChzCUmLvnRxeVPEqiSOklNAEs4cROA5UUKIf2y6TWERPEEkIHZ4SlqRRYHBQY4LdLHA4OERvXqBQWKEdDZy5G3k
KLiR49ChOi0Se+lTZ3/ku1Ue4L1QSxbLxePOkNWhjNNg7G3CQZFeSxBXxWq4nbWasbcaGE5ecjgQNYwwmXQtU9Oiu4IPYkUgksAE
PhSD7zRAXCYtscXLhUaFK22TUrktzUQbAzlTDJWUc4m6p8VSJ4TOe7ZECEBbMDbg9YLP1Tk3YqHETU1lavJvIxgyXBSz8lINMJvq
AsITD8ITZdQV34SKDVklLOCZ2EzFlZVxqCLdFoYwFtMld2SZwHBagKoW+1Vqkjv4iwnArgocGUSwteaZTJ9MwKjYlk0ofhernON7
O2mJORHnK8GIGMWfNvUnZO84d4mfXQViVpP9WR1aFS8vZLZRGnXgQ2zCcWqSkrds5Vagc2A99WA91YqRHXdvOpUQeQSKQPyu6uLA
sTgkLaT5Jko2qAwBvikDhVjnU2kZC8Cl20KLrZMOgiRH4PLg9sX033LYnWGkQAXomF1zpOl8pC2/JuNrBF8CwXN5pMVFEweQ1x4g
xYFYkqZIkFjLIVBxlBMh+zTVDLgH+dIagg6tUn8C4AsbY+g8b7JHO/5M/dCE3t8zQk78s2n5mYTucYhnK62kTjno1EilgndWkLrm
bSMHODcecGDy0LNVrZkeTmw6g5+uQoR0lWe7JY8Hm7PyKMbiUZwHgNaWrVFla44bMOE7SDy38uBG9xbjk59iieXz8jNYSiY27Bu2
p57dYcp2h1ekFqR5LjNH7g0vneTWl8KsmLypiHzx9hnPqHFzSLxbadxjpCIhMEEi+6RtABDGFOFIRJJMLqE5P3z7BRaqA7RcuYcp
EDW9G97bTHn4NFP2avRcVvHRbFvQ8AFLU/SjLwkPqFqTpbUm6lhyADsxsRho3h1qc5T+JLIXUyfGevTAwkoT0kv9iotRh3XUjr9k
6lkwpuxpDGgGXprIn034n/ZoLYHYHgI9t3pAE9QgKMpzPMaq3Uc0+r34JvQC3dWHhIIHGrK18zjQuRNdER7hg/YIkNmZZ4+QHG6Y
z8FD4Zd/Ym6dq6LTZpkIWhDkxPwbwSJT33gwVfk1jSgijNCILBiyT/i2tx8JNveVbmoFDbGL0QTdSToCYse+AbMQJNX5NEjvUq6T
ZJOSWsuhdkpwASfhSC6iL0S4Q3qsUWTZRXDbOI6gdAf5DEwTF7zG/OjyGwmXAE/4mAeQGC3nJ09KRAivx0jsGT1ltDeqDHKrp1YW
t1JfYEwhxMCEp4hkMH68aU9zeZUJp6Er8SXisaLf9meesauiXuiosZJVGaAZc9LRlMUmw+9kkfBTKs93fG0S1BAsBPnFZOAIXtQF
YhyXKQu90aWHsFolecrKHsEOwUbK5/KQWuk4gf0ftRsyVNB4i5gJtiXOFAixIZckrX8R31i+yCvtjLS8fJe+Hxu4vt4IOw87ItVh
gZbLi05yOlqfhuDCW8hhgqIKSyDXhsRZLzRJ6Q0qdFTM+xDMFCfSVn7NibMm3P2c1sy0EKpObP3C2HfdVorhbbonGtjlFb2QcmKv
v3mjS4r9BSYFBCexUV7pZRlqrHHdMOTHqtpjkpzM4eAcZ5McshpOEY4EQE0r/vkvnfYpPPuOUbMfqZptg8EAo5w6z3gOCnGRHC8i
c1+e6lw7PrnJOlX+kA0h/35VH9ncHq2zChV4iE5Zm3vtGXeug8ad63D2HMaTJITKVJlTf1RMwO4coSheh+0VLWI+mONZqxl4WRut
VFdq0qqg3tOTEPD9Lfru5Zc/vP5iIOOaocReA1q910Yqboi9s0Ey8kAyCoIkHLPCHS7epcDrDwClH/3vbz5/8ZmwV3HBiJhLrPje
AgAvvK9bQ0vucwf8sfOhKahmiL8ype28kWM9LP53HzbdyFgWkMYekGB9wUZG75J9BhH0eaQwEmcdhzvRJlOE4P2ryU8x1uzBJZCD
kdX1kSDTxcfFMtkq8VJivsQDJeLrO8ucbC/Fs7Wz6gUO9zdnfRNvfbB9BCpKSk0M/CLPmt9tazZNU8gaTUON+KANXERttsU+tPhp
SxWNKqXNCCOXd9JiV6FWpmBWhTVX5Qiq2Fn61Fv6NIj/160Jte+KVbKM4ZXUha1qkbDTx0BtK6/G3nveR5GuxGDFCC89XR5/FDFq
xpOMnvEcxI6O9aoWGDuru/ZWB/sAEn94M0QVHCOyoDTJ1RAKkHSNbLW3wrrOTJIYc4QPcTwppIl0rxxCxbEi3KgC+xqsxrAuCyJ0
aykaSXrtoS4O/UhZo0h6TQWjdRYFaWMV/eOs7MZbGZT7sUa2qbYrpCh7qo8isZcvqW+ShdhyuUtqSMVIPKotbR8ppvrQIuE4qwOr
mrVZRHhcp+ydsXMq/JCybJIOyh8d97j3EtOnRlMGiJgHSHE8ZXTiImtKJ+8Enf4EXvH387llH3tWM0flmq6NKsykkIMYSYG4TA+7
9kwL12xa8HoVKqGKxTIR4mRuEjdfF8tifYaj2Ya1kDHZMNkTEarsdLxNcZT92QU2IxjrwDaMRZ3uDjkk9M43TIzi0XiiiAFKLhBM
/xYJGaGzta0fiHLAGsQK1C3TL0VYHhKEz22zWn3XOgfiRt429AYGFgRHpDDYhXbDn5YyvaLhtijW3lzGZi6ej0t8LWA0ZmIMdmda
Vrz3uoDQSNJFfna2c+5t51y7FfQMpaIH0QLE/a7TVSYBNMbiT6QBwcwrJIRBDoOmcHpQMdq6E9QVTkOJKNeXURAGZ9fZZiPHiWEC
8fO9DvdlWlmlh6Tk3rUVhziPJ+8NfYFvqCu0cIeD6CXTXjF4YcS0BL8xaaS6Z0X54Hxf5cc19Fyh10CR/hNsLQ6K4T/FY6klh+Ky
YyblHDA5bgbSVDog83Dr466Lwj23cIHegTEIV+n1QlUPjb5bkpamA8lMiC4bEo14cMlE4guxGSfgpe6Py2NJT9BdjNNVsFhbDVFl
RQ6FpKkhg1RyVmZNgJxYErna3oVGcxPWaFA+NtlV3TtLTYDUwMoKiWgQLel9rIpVB/1eZF7iD5G8ujtHztsdPs1KoBLqiGyAUFKg
mQdxz7yQse4dhFUV4VQHBsA3nlpyE1RLbsJqCec50ty4gI9m7Mw8Zyp70Rpn5I0zCo4TdInqAIDWMtUwVquinK2vYI6aNXPwlcCi
b7mi04Ck703ArsftdH9nP4LJZWTLgOmaR+OYbTWS0rIsCI09CEHQnxvExclqW9zzKLCywSV0J0/SO6yyoXese0fqk1sTvGOVAbcc
0MWeEwTEx8ApPfumnJKz0om3Uoj8ExPngTrmqgKDXhWsHYRSXO3ocllB8Z2p4wKiW5Nj88zaumcMq2fWyp4xcdCEuA6clak376ky
falZWoqIBBpxD6pHyLZyTlW5e5NbqeRGnnBgdUHxfy8Hvg0hdJlvEXLvrNeUaSVq9tZraqml8KNimE7b8LAucsVIaPsa6enHiH9Z
RivpbU9AGykFqOYCTL+z23s1f5lTUShtQcKG1eycDbr2Nug6SGRu2nVdgoSycUDbHXPkoCmHoipBAW9cr74qHQBketZp6/lkdYwC
AX+wY4OlzoJvvAWzmxEbGzEGVJEkFI21m1/pFhCm1qghIVcgDNwTrtuOVALKJQRnYXagNrYfTTg0ejJs1EOLc6280u+OuOCV+3bL
Mbs1YldS6ntlNJ6mRLs7rTJBDAmklriJ8VwNJLRcauG/i25vI4nFM2SIEzE55OjIhXabCgupKX5obcXM2wpoJ5Oh5jKA5PWNPus3
84baISVB1OEE7vFtnpoabkRbstIko5ZcHECILM2FxNuUmVdgf4JayaEgpWOZpwt1zCCP8sUJKDks9TlCtT7GqgRmsQ+cWF7dYjok
8I/oz3AxlP+Y2w779CkQnH7jif43LPqPBlOiDHkOsUxPVXGy6J8Q4UQjL1D2SpdnVMHTqMDGWjoHcOpgDSjgiYJeAEYtMdCIFCKO
9SE1+jOjRX/WpkR/Ftah9cb/ThWa1OeFauIxYpPArIx9TZi0o0O70t7QF/eGimJYBjXWmPuBrth/QNqihb4XgJ21pJCSylGmi8NK
6ujEIo6pki2CKX3J1TmkJSoVJU5I3cyTU2csp6Ij/X5gIuFg3HW6r9LFdtmNeUyF/dTVkOXYmJFWS13ikZtFy3Ntvskb7YXZtsvF
TvAYhFCliajDoPuTvwjlUi9y9YtEHwBn5SNv5SMmLMjJ3CY8rz5jn2hRimxx7VDJi8KGDQPQCZvVS/gw+rQjeuI0Es9cIMDTnKlv
zizH3iwhvU6tNStnX6O+o8LIlnS4qkFcti8eCefOgflO2kOrZ6ai88zxQ8TmypmYTX11KkYMqyonPLZckhNHgHMNrTVNvDVNGOdU
A46HT8Tiz85nltB5kEHEBWolJtvURzih7OQhKau08XVXdVKGUDco0krjBXJbmsCKkN+BPXRKJcViuySoTlHVifkbIu3EuF0/EVtt
D6bcyqo4OHt3IMKguKaXIQYiIvLZ/mzF6DpQnXpQnTKmRConRkl30ytWW+WUsTYgcUEOlkQIZ68IWXUIa4nP6kwFoHrdUiFe1urW
LHS8RBLSaPsFlfBseXo6ytfUWeYFTdr2Ht677rFYAoiUC46/NC47FWEMHaFhcPtifwXLRLttz1mFXBkgor8joZsQeBAFfgshjaUp
Iubs07W3TyFhetYSn0fTYTrrBx3qRRtf5Pt/E6wqR8tz6O3OqiAYlFEPaZw13nhrvAmucdbqgcnWj5gEEbv5DbWNR4vJkGjQeLiI
Y5y4GQlDo3vNqY0g1fgKntxZp3J9gmwpuajnBEk7+nv0iL/eCeQ8ZYnEOdImW+XsnYXPvIUH6ikzkZrq1pxuJz8xQwgcr7kXCSbB
hbN+9PuJldQIRfxhSDKbLax7feaXchmKCUMqo78hmYwfe4GcHDCOKEzSj+z4UG1QseK1LxUL8auuEbymSuXxdyOxg6CNOE4SEqoD
/7kH/zlXItUN+UYM1HWV2ez4PiP/XgF9f8AJhhIRjrlAVyCMdmZJuxIJmazXTQCRfUvb+wsOavimiQ3lflQsJR8DJIJodaotc8QO
oOKv3G8TpqujdFmgz9dWeFTfhJuK7R08U+Hi0JURh76QCOn2tWrIhiXRCvZ83HQmK+lYdqC9qoqeuldTMC1mIAvTcdVEtQfpOgD8
edh4DHvAOv0DuR7KcuO/4Vz4GM4KEZnrpMy/IuDbCQhzT7qes3Q91TRJ2UipzXMVjadVDh1MLRFXcseLrlTJOXQ72DcCEAnak0VR
4rK2EpQsVaO9q3p0xVy7ZDKNo4O/nMJKhG60YT+GLnixxuK7vLpW7BxTCR7EvqNEJSlYte8CVbLnnrg+Dxq652FDN6+X5lSxCtq7
iHH9k7dgjRFaGwqOVaHUqGgjZiJnIWNvIZDoX9sRxCawU3DCBH6qzHLLdiv2YAjAHJ7JLjpdUyyAHUE7tUpqPotqrdK2974fwLrm
B6XNtS1Ay5Ec+JoTluoiIK2jACk62kDWsS7+6bTghTDNoU7OVpejRKsEt7s4JfnmnmIxD5YAnIeVAMYRooVyPxEXU9drhiFL6UEM
oQZBuGK23BxoltHEhNx1PseIUMYtaP6lSXtSVZ3ce0blataLq1she8nO6FtbBfU0YFVokHUhihhcHPhMPfhARXih/dt5UbxFnhNb
iIgr/vOs79ARiwVbyRBBekLslslzYp4p0QJWlABNAZEfZ9S0u9yBoMLw083ccaDeGAcqtFl8R3D2cbcP+k9viNPjTjmnknBlKgUZ
U49KRTZxTHytnAOkaw9I10EkurGrxlTEoTjtDfKyZU49lXLXWEmAKXYD6iA55vWi3G+7aNIb7IuS9PcuR93eMt/haHZaRrhSjHuH
Wpk9dhey8p8BsVPZFHg096x5LzR1hn7CMd5XJDzDbcbGhO5Z+YfkljVlPtm3loBsm4jRlWQPr/CM73Dju/n+GnGsEwranLiGtZMv
gVBnERudAHXeTmefbrx9ugnuk9ExTgsRn+25he6ZHA3tUj+6lfy9UgHnT8IYb/bak/r2B3mTb/3AJYh/V00AkhEMuzWOPqEm16PG
gCyS8aSdnWh6HKAmLhGj2yiWikNzE4KmQ9BFi5FrXNwwfweoMw+o0F9e2tvD8czIaYFVAIlDEhd4IukNYYJQkiQsHTdKIOwC+KTd
jmfQGSvFmEjOgah9Eylv7j8KEI3wNTG8B6Rro4jUU9vioKSpfcnClCq0amNzpHHTAc7cAw5fH3OsCzS0/HCi6x2XJkOdRuCVq8I+
g+iFXcQWXOBAi+YL2qBXtxXYmVvaxmNJIgmoCy99gKg9SHdCAN3faPmCjqfmfQ0KfQum/Mpa8PUEobq/HDMSwhkkt+zm5R4K00Fx
IIox+P6rL0h9LFFyo7w1Gox3oMbOgSp4bEn/732qetvvB9qcl+SDqtjUMAV2r2iFuCSxxBkh0iVXYPQGSMAFE0IHhVw20AvVMlNo
IWPo7AHdG1HjmjgDdbc/7g7nbs+UgGJ7Nc06ouUFTqMu04Mo7Gaqu96fq5uyTEqVpsRE0Uxa8pDu/EQkuczXSB3m8buMzl/dEaFW
ZFXa9yZvKXl8yDO+oWZ6keOkr12objvPw7VSPHkvkPIUrgXXnvo095XKuSiVuqbIfKbpizjNmAS5t76pio6x9YJ1dZC+aMZO4hCh
CgklWIvOSz5lHO+llNNmIfBTsLkQLpZDug/eFDgMK6E4x7bd00rgRr3FvnU/y3igLaFaORpHn5DicBWNwQv1PXmdf2cMVZZ7xmPI
hYIxZ0/N9DD6mjz79qehf/PmMHz15rDt7s30oEpwVeqWPCkHizFFr+qijGxPK1ffv/z24nq87iMm+sjg6LmTG/mTGylptiL6bSed
axA217LhKHixhlD+1GU1pNCboshozyYKGq9xL3C1+JA7hWYR9v/oFovH0ZN7JjXQpNpnR6/9FuEGdURfb4MpUuHOuXChTtJWI03Z
xUA7gdu7cJstl8WZKqFwbcrBVi6oxz6ox2E8mDjCFi6ZVeKTuZ3Xuu8Zey+rd3BhYeNC6yXOeO5e4SyFqANVsRa00VosgyKr87A2
Eed6M5p5KNcnqMthRJKaxIMizyY3GcQXqa/KbQ/FXwaUtTfBKBK4JMWJjUIlnUGtkiBUx59PgJ/4gJ9YdW6Ho+hRjvHkWo7IaDDv
q7hqlFGJ1YA6JFaKTsFaJ3djNll4CW5gxITiEEobhfm7hX2x9uK7V6+/Wrx6jSRI/vjND2+8a7XR6GunEdJvP44mSMW94o/XPcu2
DTmW5gg0QVwJ/WHBsUsD/93qg8s9rUiCSXSZ44oaJr37HtOX/1R+C6J00V9uaQK9nndxsjjDYOY4DIgYVIdklXISjlyddchI3SQM
IGGXYXYrqYtSMM5vdiUuu0kcyHHfaWNYcyWfU3O9CdE15h9WuYSdJRy1wiEZm03lxMUQbkx93GBtXm/kyPSoLCcS5SGGxVF/Ph0x
FWTJlMVPBNIxXUTcNyf+i3kzy00mRcLmilCBaxo9XI+jHC1QgfEOV1gL8f+Y/mPw9wWUGovuFcXTFfvYGUxYrb/rWySzPWRmKZvf
CQ5Ip+N3jEjIeu/oAXqA0nXYEaSvfUhfK6+OzEXV0iRon5VlFGXpTRFbyBrNKyWb6UYoL4M74hQC4Eyq6DBWDHI5twEwG7tDrUgr
KZaP3Q+DFycDp+t3gEWq1Qacj30ZlO+dSfLNidarg+qJPV6LXybk0gEMnMtVUoseNeVrSbIGBCDYhHaWtKljpc/7L2XdrY3e3JR6
JBmwqO2Kdlyb7ffJ5hNYdekf6yKzFSevdD+GtfxjA0uSJG5nw/Yyhr8yz7gdP9kPIN3SkS5keNEP3EsHKNldrODzrCSFvsvFKagr
DaBNlue36g5cr2PCgNpch+GJ+tlOSJkt3F/+llSHdFV3O1wv1dUW6qym2XQUGdE1/9JTY3xLohVPmY0CT10wPPSv4R2yeecF0pAT
9h6tjuaeOmsIdVmtDEJIICBp8uNvjP0v52uIQyespYrtApYPpGrKDVQDiWA9Rf/v/0aPfJ7oWMtOSCGGWmI5gpXYF7tsLV3BXENd
8QUJl90hsbUXSv+C5LrM7dJNdk0sOVd8m7yf9KJUKM9vSvCe+fDWnmtVk9LcBWxfFGMV/UmRI84lHQfRq43cXSS2Il8CV31Iv0bv
cueu78JBAmITx2RNPrRxc9/UF5I6H0m2I9L3aEmej6rktwQ69Fy507rXSAnqEDPvhvchSTMnrmmJmhKb6AmchLRdnoQpPUxSSki4
tHdn7u+Ovp6iubHxkS2nhu3YDzQLRaEBnbj7Fql7VvFKflESAuTCGWZHeLtSm2Jd3m14sjp1ck9yIKsyjlvCFokurRG6gFhP5Vr5
4fUrK6ThIuvp82++/qJh2PoKhUuvB1zZTfhVA8TY12ZjFes40/cBqHtWqkRbGkLLCau62+Qgq1Hpnxxwc2+Xoz4t9khJdZ/rorxy
KZgOhLa3b6Oj3+2FjPyFjHStNV5FY49ty37TtScq6/JqS8hTg1uXtqlMeLlDx4mAD8Fo3J4IoHxdbsy+nQzgBuibFBJCT7nxISwb
cJLuW76uUSAs9TOb3JqhKbpuqhwqP6sybbkQHvsQhsI7MpX+xq77K7G2S93sBiOTBh7TQkyc7+5ZiaOU2odAN2k/LUv7sPh+yD9z
VHheJp/TBcDEB8CE7/0e6hs4R+rqDLcKI9RKuZp7/QE7SdxS+hBaH1+culg2kS/9oPsVqyRt8g6WMxMJxiEswWxIVVLtQmU/5vkC
Lks3/6o1CHwnIfNiLePxXMBNfcBBK7N7VuUlJ8AARhtdVpLDR0YhUF230hzgRAg2jAIEGQYO7qRofccCoEQa2EXypbKUu8Brf4HX
+kZ4HSOiC9mBrYC0MeMj/DEw1GccSRdN0d615pEBCNy0H4tD4AYZ/3i0nY8P3SdDy73xl3vj3CijeYeq+iPZ7yjQUinljwutJxwS
hfOmoMAFklHEQteKWpm0PoF5CAizdiC4ZtYPEImzC4SL+707LTGZe+wRclMRHKH0Cyh0cvUq0YbMVDdK1410SA0+ZfMpPZY++N5H
G8gzH8ha+HTpy7+eVZaXWJtlkcZSNkjFPAop686hk/vNpJZWHW2OJcKeQ0CetxUKQko+uKknc19UZ3k61dV0Yt9PZuczuoCZ+4CB
3OdHAUk5gG1qXREphxhJ+6m+/lMF1bDAvbdqr7K5R8t6u0H0LU+GQ284RLJJ3GwybJSQuWerOXoICn52BcPFKVzD8Kk4K9gIu3JP
GY6btZ1yBbCOwxoNw/K4KtK6k1LHfCh4JuGs5iajWRXF1rcY45140pTOxNoRHNdybv6QA/Cx0rZRiT1w3IL0EAEHoxtW45W4bL7b
ar32/pmrDyTh3+tNdbNs68W/haGJcZQsQ39yn6hu7kKgDQslLSMbs4QMzKY/Kx3w0osZ9mG6Bg00rnjZd7UzFaF8Fy5M2cMnb4Bw
DR1v/EISwgCZBzx1+TxUkeGFLoJ0hS859pZIJVegnTVJPaZ0xzpbNzejC88xEbMq+9SU6jBlb6h5TtolhJVPjbNLvV5xlT9i/bv8
jPoM/x/bXi0L
""")).decode(), globals())


This notebook trains a model that reads a support ticket and decides whether it is urgent or routine. It uses the same tickets as the video. Two of them are one word apart, "the app is not working" and "the app is working", and a model that counts words gives both of them an urgent score. By the end you'll have built every piece of the classifier yourself, and one new kind of feature will pull the two tickets apart.

You'll work through the same ideas as the video, in code:

1. Score, prediction and margin
2. The zero-one loss
3. The hinge loss
4. The logistic loss
5. The update
6. Word features
7. Bigrams
8. Sparse vectors
9. The classifier in code
10. What a line cannot do
11. The payoff

Every section starts with a short explanation and a worked example you can run. Then come exercises that build on each other. Work them in order.

In [ ]:
#@title Helpers and imports
import math

import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
torch.manual_seed(0)
M_AXIS = np.linspace(-3, 3, 601)


def draw_margin_axis(ax, margins, names, title=""):
    """Tickets as dots on the margin axis: mistakes on the pink side, correct answers on the green side."""
    ax.axvspan(-3, 0, color=PALETTE["pink"], alpha=0.12)
    ax.axvspan(0, 3, color=PALETTE["mint"], alpha=0.12)
    ax.axvline(0, color=PALETTE["ink"], lw=1)
    for k, (m, n) in enumerate(zip(margins, names)):
        ax.scatter([m], [0], s=90, color=PALETTE["mint"] if m > 0 else PALETTE["pink"], zorder=3)
        ax.annotate(f"{n}\n{m:+.1f}", (m, 0), xytext=(0, 12 + 22 * (k % 2)), textcoords="offset points",
                    ha="center", fontsize=8, color=PALETTE["ink"])
    ax.set_yticks([]); ax.set_xlim(-3, 3); ax.set_ylim(-0.5, 1.2)
    ax.set_xlabel("margin = label × score")
    if title:
        ax.set_title(title)


def draw_losses(ax, which=("zero-one", "hinge", "logistic", "squared")):
    """The classification losses as curves over the margin."""
    curves = {"zero-one": ([_zo(m) for m in M_AXIS], PALETTE["ink"]),
              "hinge": ([_hinge(m) for m in M_AXIS], PALETTE["amber"]),
              "logistic": ([_logistic(m) for m in M_AXIS], PALETTE["cyan"]),
              "squared": ([(1 - m) ** 2 for m in M_AXIS], PALETTE["violet"])}
    for name in which:
        ys, colour = curves[name]
        ax.plot(M_AXIS, ys, color=colour, label=name, lw=2)
    ax.axvline(0, color=PALETTE["ink"], lw=0.6, ls=":")
    ax.set_ylim(-0.2, 4.2); ax.set_xlabel("margin"); ax.set_ylabel("loss"); ax.legend(fontsize=8)


def draw_weights(ax, w, title="", top=None):
    """A bar per feature weight: amber pushes toward urgent, violet toward routine."""
    items = sorted(w.items(), key=lambda kv: -kv[1])
    if top:
        items = items[:top] + items[-top:]
    names, vals = [k for k, _ in items], [v for _, v in items]
    ax.barh(names[::-1], vals[::-1], color=[PALETTE["amber"] if v > 0 else PALETTE["violet"] for v in vals[::-1]])
    ax.axvline(0, color=PALETTE["ink"], lw=0.8)
    ax.set_title(title, fontsize=10)

Here are the two tickets from the start of the video. The model below was trained on 13 old tickets by counting words, and it gets all 13 right. Both new tickets still land on the urgent side.

In [ ]:
COLD = [("the app is not working", 1), ("the app is working", -1)]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={"width_ratios": [1.3, 1]})
for row, (text, label) in enumerate(COLD):
    left = 0.0
    for word in text.split():
        wt = _W_UNI.get(word, 0.0)
        axes[0].barh([row], [wt], left=[left], color=PALETTE["amber"] if wt > 0 else PALETTE["panel"], edgecolor=PALETTE["night"])
        if wt != 0:
            axes[0].text(left + wt / 2, row, word, ha="center", va="center", color=PALETTE["night"], fontsize=9, weight="bold")
        left += wt
    axes[0].text(left + 0.03, row, f"score {left:.1f}", va="center", color=PALETTE["ink"])
axes[0].set_yticks([0, 1], [t for t, _ in COLD]); axes[0].set_xlim(-0.1, 1.4); axes[0].axvline(0, color=PALETTE["ink"])
axes[0].set_title("Word weights stacked into a score")
scores_cold = [_dot(_W_UNI, _phi_uni(t)) for t, _ in COLD]
axes[1].bar(["not working", "working"], scores_cold, color=[PALETTE["pink"], PALETTE["cyan"]])
axes[1].axhline(0, color=PALETTE["ink"]); axes[1].set_ylim(-0.6, 1.2)
axes[1].set_title("Both scores above 0: both called urgent")
plt.tight_layout()
plt.show()



## <font color="#ff2e88">Score, prediction and margin</font>

A help desk gets thousands of tickets a day, and we want a program that marks each one urgent or routine. The training data is a pile of old tickets, each with the decision a person made about it.

That decision is the **label**: the right answer for one example. For the arithmetic we write urgent as $+1$ and routine as $-1$. A task with two possible labels is called **binary classification**, and a predictor that outputs a label is a **classifier**.

The features are word counts: every word gets a feature holding the number of times it appears in the ticket. Section 6 builds them properly. For now we store one ticket's features as a Python dictionary from word to count, so `"refund question"` becomes `{"refund": 1, "question": 1}`.

A trained model has a weight for each word. Here "crashed" has weight 1.2 and pushes toward urgent; "thanks" has $-0.7$ and pushes toward routine. A word with no weight adds nothing.

The **score** is the dot product of the weights and the features, the same sum of products as in linear regression:
$$\text{score} = w \cdot \phi(x) = \sum_j w_j\, \phi_j(x).$$
The **prediction** is the sign of the score: positive means urgent, negative means routine. A score of exactly 0 sits on the boundary, and we will count it as a mistake.

The size of the score says how firmly the model believes its answer. A score of 1 is comfortably urgent; 0.1 is barely over the line.

Real classifiers usually add a constant feature of 1 for every ticket, whose weight is the **bias**. It shifts every score by the same amount. We leave it out to keep the sums short.

The **margin** is the label times the score, $m = y \,(w \cdot \phi(x))$. A positive margin means the prediction was right, a negative one means it was wrong, and its size says by how much. With the margin, a larger number is always better, whatever the label.

In [ ]:
W1 = {"crashed": 1.2, "refund": 0.9, "again": 0.5, "thanks": -0.7, "question": -1.0}
TICKETS3 = [("app crashed again thanks", 1), ("quick question thanks", -1), ("refund question", 1)]
PHI3 = [{"app": 1, "crashed": 1, "again": 1, "thanks": 1},
        {"quick": 1, "question": 1, "thanks": 1},
        {"refund": 1, "question": 1}]
Y3 = [1, -1, 1]

fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.2))
for row, (phi, (text, y)) in enumerate(zip(PHI3, TICKETS3)):
    s = sum(W1.get(k, 0.0) * v for k, v in phi.items())
    print(f"{text!r:30} label {y:+d}  score {s:+.1f}  margin {y * s:+.1f}")
    left = 0.0
    for word, count in phi.items():
        wt = W1.get(word, 0.0) * count
        if wt:
            axes[0].barh([row], [wt], left=[left], color=PALETTE["amber"] if wt > 0 else PALETTE["violet"], edgecolor=PALETTE["night"])
            axes[0].text(left + wt / 2, row, word, ha="center", va="center", fontsize=8, color=PALETTE["night"], weight="bold")
            left += wt
axes[0].set_yticks(range(3), [t for t, _ in TICKETS3]); axes[0].axvline(0, color=PALETTE["ink"])
axes[0].invert_yaxis(); axes[0].set_title("Each word's weight, stacked")
draw_margin_axis(axes[1], [y * _dot(W1, phi) for phi, y in zip(PHI3, Y3)], ["crashed", "question", "refund"], "The three tickets on the margin axis")
plt.tight_layout()
plt.show()



<font size="4"><b>Labels as numbers</b></font>

Write `to_label(decision)` that returns `1` for `"urgent"` and `-1` for `"routine"`. A `match` statement reads well here.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

A `match` statement on `decision` reads well here, with one case for each word.

</details>

In [ ]:
# Exercise 1.1: running this cell checks your answer
def to_label(decision: str) -> int:
    ...  # your code here



<font size="4"><b>The score</b></font>

Write `score(w, phi)`: the dot product of a weight dictionary `w` and a feature dictionary `phi`. A word with no weight counts as weight 0. Return a plain float.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Loop over `phi.items()`. `w.get(word, 0.0)` gives 0 for a missing word.

</details>

In [ ]:
# Exercise 1.2: running this cell checks your answer
def score(w: dict[str, float], phi: dict[str, float]) -> float:
    return ...  # your code here



<font size="4"><b>The first ticket</b></font>

Use your `score` to compute the score of "app crashed again thanks" with `W1`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Its features are `PHI3[0]`.

</details>

In [ ]:
# Exercise 1.3: running this cell checks your answer
score_first = ...  # your answer



<font size="4"><b>The prediction</b></font>

Write `predict(s)` that returns `1` when the score `s` is above 0 and `-1` otherwise.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

One line with a conditional expression.

</details>

In [ ]:
# Exercise 1.4: running this cell checks your answer
def predict(s: float) -> int:
    return ...  # your code here



<font size="4"><b>The margin</b></font>

Write `margin(y, s)`: the label times the score.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

One multiplication.

</details>

In [ ]:
# Exercise 1.5: running this cell checks your answer
def margin(y: int, s: float) -> float:
    return ...  # your code here



<font size="4"><b>Three margins</b></font>

Make `margins3`, the margin of each of the three tickets with `W1`, in order. The margin axis appears when the check passes.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Walk the tickets and their labels together; `zip` pairs them. Your `score` and `margin` do the rest.

</details>

In [ ]:
# Exercise 1.6: running this cell checks your answer
margins3 = ...  # your answer



<font size="4"><b>Which are mistakes</b></font>

Make `mistakes3`, a list of 3 booleans: `True` where the ticket is classified wrongly. Count a margin of 0 as a mistake.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

A mistake is a margin at or below 0.

</details>

In [ ]:
# Exercise 1.7: running this cell checks your answer
mistakes3 = ...  # your answer



<font size="4"><b>One score, two labels</b></font>

A ticket gets a score of $-0.8$. Give its margin if the label is urgent, `m_urgent`, and if the label is routine, `m_routine`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Multiply the label by the score.

</details>

In [ ]:
# Exercise 1.8: running this cell checks your answer
m_urgent = ...  # your answer
m_routine = ...  # your answer



<font size="4"><b>How far to flip</b></font>

The refund ticket scores $0.9 - 1.0 = -0.1$. Holding "question" at $-1.0$, what weight of "refund" puts its score exactly at 0? Store it as `refund_flip`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

The score is the refund weight plus the question weight.

</details>

In [ ]:
# Exercise 1.9: running this cell checks your answer
refund_flip = ...  # your answer



<font size="4"><b>Add a bias</b></font>

Add a feature `"bias"` with value 1 to every ticket, and give it weight 0.2. Make `margins_bias`, the three new margins.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

`dict(d, key=value)` copies a dictionary with one more entry. Give every ticket the bias feature and give the weights the bias weight, then reuse your margin list.

</details>

In [ ]:
# Exercise 1.10: running this cell checks your answer
W1_bias = ...  # your answer
margins_bias = ...  # your answer



## <font color="#7b2ff7">The zero-one loss</font>

The loss we care about in the end is simple: did the model get the label wrong? Count 1 for a mistake and 0 for a correct answer. This is the **zero-one loss**:
$$\text{Loss}_{0\text{-}1}(m) = \begin{cases} 1 & m \le 0 \\ 0 & m > 0. \end{cases}$$

Drawn against the margin, it is a step. Averaged over the training tickets it gives the **training error**, the share of tickets classified wrongly. One minus the error is the **accuracy**, the number people usually quote.

Gradient descent needs a slope, and this loss is flat almost everywhere. At the refund ticket's margin the ground is level, so the gradient is 0, and an update moves nothing.

The same holds from the side of a single weight. Vary only the weight of "question" and count the mistakes on the three tickets: the count is 1 at $-1$, drops to 0 past $-0.9$, and climbs back to 1 at $0.7$. Between those jumps it is flat. The slope is zero on the flat parts and undefined at the jumps, so it never says which way to go.

The count also treats a near miss and a bad miss alike: margins of $-0.1$ and $-5$ both cost exactly 1. And finding the weights with the fewest mistakes is a hard search problem in general; for large data sets no fast method is known.

So we report the zero-one loss and train on a stand-in: a loss that has a slope and still rewards a large margin. A stand-in like this is a **surrogate loss**.

In [ ]:
WQ_GRID = np.round(np.linspace(-2, 2, 401), 2)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
draw_losses(axes[0], ["zero-one"])
axes[0].scatter([-0.1], [1], color=PALETTE["pink"], s=80, zorder=4)
axes[0].annotate("refund ticket: flat ground", (-0.1, 1), xytext=(-2.8, 2), color=PALETTE["ink"], arrowprops={"color": PALETTE["ink"], "arrowstyle": "->"})
axes[0].set_title("The zero-one loss is a step")
axes[1].step(WQ_GRID, [_err_q(v) for v in WQ_GRID], where="post", color=PALETTE["pink"], lw=2)
axes[1].axvline(-1.0, color=PALETTE["amber"], ls="--", lw=1)
axes[1].set_xlabel("weight of 'question'"); axes[1].set_ylabel("mistakes on 3 tickets")
axes[1].set_title("Flat, then a jump, then flat")
plt.tight_layout()
plt.show()



<font size="4"><b>Count a mistake</b></font>

Write `zero_one(m)`: 1 when the margin is at or below 0, and 0 otherwise.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The same comparison you used for `mistakes3`, turned into a number.

</details>

In [ ]:
# Exercise 2.1: running this cell checks your answer
def zero_one(m: float) -> int:
    return ...  # your code here



<font size="4"><b>Four margins</b></font>

Make `zo_vals`, the zero-one loss at the margins 1.5, 0.05, 0 and $-3$.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Your `zero_one` in a list comprehension.

</details>

In [ ]:
# Exercise 2.2: running this cell checks your answer
zo_vals = ...  # your answer



<font size="4"><b>Training error and accuracy</b></font>

From your `margins3`, compute the training error `error3` (the mean zero-one loss) and the accuracy `accuracy3`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The error is the share of the 3 tickets that are mistakes. Accuracy is the share that are right.

</details>

In [ ]:
# Exercise 2.3: running this cell checks your answer
error3 = ...  # your answer
accuracy3 = ...  # your answer



<font size="4"><b>Accuracy in NumPy</b></font>

With many tickets the same count is one line of array code. Given the arrays `SCORES8` and `LABELS8` below, compute `acc8`, the share of tickets with a margin above 0.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Multiply the label and score arrays element by element and compare with 0. `np.mean` of a boolean array is the share of `True` values.

</details>

In [ ]:
# Exercise 2.4: running this cell checks your answer
SCORES8 = np.array([2.1, -0.4, 0.3, -1.8, 0.0, 1.2, -0.2, 0.9])
LABELS8 = np.array([1, -1, -1, -1, 1, 1, 1, 1])
acc8 = ...  # your answer



<font size="4"><b>Mistakes as one weight moves</b></font>

Write `errors_at(wq)`: the number of mistakes on the three tickets when the weight of "question" is `wq` and the other weights stay as in `W1`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Copy `W1` with the question weight changed, then count the mistakes with your earlier functions.

</details>

In [ ]:
# Exercise 2.5: running this cell checks your answer
def errors_at(wq: float) -> int:
    ...  # your code here



<font size="4"><b>The staircase</b></font>

Make `err_curve`, the number of mistakes at every weight in `WQ_GRID`. The plot appears when the check passes.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

A list comprehension with your `errors_at`.

</details>

In [ ]:
# Exercise 2.6: running this cell checks your answer
err_curve = ...  # your answer



<font size="4"><b>Where the jumps are</b></font>

Work out by hand where each jump happens. The refund ticket scores $0.9 + w_q$, and the routine ticket "quick question thanks" scores $w_q - 0.7$. Give `jumps = (a, b)`: the weight where the refund ticket's score reaches 0, and the weight where the routine ticket's score reaches 0.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Set each score to 0 and solve for $w_q$.

</details>

In [ ]:
# Exercise 2.7: running this cell checks your answer
jumps = ...  # your answer



<font size="4"><b>The slope is zero</b></font>

Measure the slope of the mistake count at $w_q = -1$: move the weight by 0.01 and divide the change in `errors_at` by 0.01.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Call your `errors_at` at both weights.

</details>

In [ ]:
# Exercise 2.8: running this cell checks your answer
slope_count = ...  # your answer



<font size="4"><b>Every combination</b></font>

One way to find the fewest mistakes is to try every combination of weights. If each of `d` weights may take `k` values, write `combinations(k, d)` for the number of combinations, then compute `combos_words` for 10 values per weight and the 39 words of the training tickets.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Each weight multiplies the number of choices by `k`.

</details>

In [ ]:
# Exercise 2.9: running this cell checks your answer
def combinations(k: int, d: int) -> int:
    return ...  # your code here


combos_words = ...  # your answer



## <font color="#00a383">The hinge loss</font>

The **hinge loss** is the larger of two numbers, 0 and one minus the margin:
$$\text{Loss}_{\text{hinge}}(m) = \max(0,\, 1 - m).$$
Against the margin it is a straight line sloping down, which meets the floor at margin 1 and stays there.

The hinge is never below the zero-one step. Wherever the step charges 1, the hinge charges at least 1. A loss that sits above another everywhere is an **upper bound** on it, so pushing the hinge down also pushes the mistakes down.

It charges a correct ticket with a margin between 0 and 1. A margin of 0.1 is a guess that a small change to the weights would flip, so the hinge asks for a margin of at least 1: a safety gap on the right side of the boundary.

Left of 1 the slope with respect to the margin is $-1$; right of 1 it is 0. At margin 1 there is a corner where the slope jumps. A corner has no single slope, so we use the value from either side. A slope chosen this way is a **subgradient**. We use 0 at the corner.

For the three tickets, the first two have margins of 1 or more and pay nothing. The refund ticket at $-0.1$ pays 1.1, so the average is about 0.37. Seen from the weight of "question", the average hinge loss now slopes toward a valley. Like the squared loss, it is **convex**: every chord between two points of the curve lies on or above it, so there is one valley and no higher dip to get stuck in.

A linear classifier trained with the hinge loss is called a **support vector machine**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
draw_losses(axes[0], ["zero-one", "hinge"])
axes[0].fill_between(M_AXIS, 0, [_hinge(m) for m in M_AXIS], where=(M_AXIS < 1) & (M_AXIS > 0), color=PALETTE["amber"], alpha=0.2)
axes[0].set_title("The hinge sits on top of the step")
axes[1].step(WQ_GRID, [_err_q(v) / 3 for v in WQ_GRID], where="post", color=PALETTE["pink"], lw=1.4, label="error")
axes[1].plot(WQ_GRID, [_hinge_q(v) for v in WQ_GRID], color=PALETTE["amber"], lw=2, label="average hinge")
axes[1].set_xlabel("weight of 'question'"); axes[1].legend(fontsize=8)
axes[1].set_title("The hinge slopes toward the valley")
plt.tight_layout()
plt.show()



<font size="4"><b>The hinge</b></font>

Write `hinge(m)`: the hinge loss at margin `m`, as a float.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`max` of two numbers.

</details>

In [ ]:
# Exercise 3.1: running this cell checks your answer
def hinge(m: float) -> float:
    return ...  # your code here



<font size="4"><b>The three tickets</b></font>

Make `hinge3`, the hinge loss of each of the three tickets, and `avg_hinge3`, their mean.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Apply `hinge` to your `margins3`.

</details>

In [ ]:
# Exercise 3.2: running this cell checks your answer
hinge3 = ...  # your answer
avg_hinge3 = ...  # your answer



<font size="4"><b>An upper bound</b></font>

Check on `M_AXIS` that the hinge is never below the zero-one loss. Store the result of the test as `bound_ok`, and the smallest gap `hinge - zero_one` over the grid as `min_gap`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Build the list of gaps first, then `min(...)` and a comparison with 0.

</details>

In [ ]:
# Exercise 3.3: running this cell checks your answer
gaps = ...  # your answer
min_gap = ...  # your answer
bound_ok = ...  # your answer



<font size="4"><b>The slope</b></font>

Write `hinge_slope(m)`: the slope of the hinge loss with respect to the margin. Use 0 at the corner $m = 1$.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Two cases, split at margin 1.

</details>

In [ ]:
# Exercise 3.4: running this cell checks your answer
def hinge_slope(m: float) -> float:
    return ...  # your code here



<font size="4"><b>Check it with a nudge</b></font>

Measure the slope of `hinge` at margin $-0.5$ with a nudge of 0.001, and store it as `nudge_hinge`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

The change in the loss divided by the change in the margin.

</details>

In [ ]:
# Exercise 3.5: running this cell checks your answer
nudge_hinge = ...  # your answer



<font size="4"><b>The hinge from one weight</b></font>

Write `hinge_at(wq)`: the average hinge loss on the three tickets when the weight of "question" is `wq`. Then make `hinge_curve` over `WQ_GRID`. The plot appears when the check passes.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

It has the shape of `errors_at`, with the hinge loss in place of the mistake count and an average in place of a sum.

</details>

In [ ]:
# Exercise 3.6: running this cell checks your answer
def hinge_at(wq: float) -> float:
    ...  # your code here


hinge_curve = ...  # your answer



<font size="4"><b>The floor of the valley</b></font>

From `hinge_curve`, find `floor_val`, the lowest average, and `flat_range`, the smallest and largest grid weights whose average is within `1e-6` of the lowest.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Find the lowest value of `hinge_curve`. A boolean mask on `WQ_GRID` keeps the weights whose loss is within `1e-6` of it, and the range runs from the first kept weight to the last.

</details>

In [ ]:
# Exercise 3.7: running this cell checks your answer
floor_val = ...  # your answer
flat = ...  # your answer
flat_range = ...  # your answer



<font size="4"><b>Why the floor isn&#x27;t zero</b></font>

The routine ticket has margin $0.7 - w_q$ and the refund ticket has margin $0.9 + w_q$. Give `margin_sum`, their sum, which doesn't depend on $w_q$.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Add the two expressions; the $w_q$ terms cancel.

</details>

In [ ]:
# Exercise 3.8: running this cell checks your answer
margin_sum = ...  # your answer



<font size="4"><b>The chord test</b></font>

A function passes the chord test if, for any two points $a$ and $b$, its value at the midpoint is at most the average of its values at $a$ and $b$. Find a pair `bad_pair = (a, b)` where `zero_one` fails it. Then store in `hinge_passes` whether `hinge` passes for every pair in `PAIRS`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Pick $a$ and $b$ on opposite sides of 0 so that the midpoint lands on 0. For the hinge, test the midpoint inequality on every pair with `all(...)`, and allow `1e-12` for rounding.

</details>

In [ ]:
# Exercise 3.9: running this cell checks your answer
PAIRS = [(float(a), float(b)) for a, b in np.random.default_rng(3).uniform(-3, 3, size=(500, 2))]
bad_pair = ...  # your answer
hinge_passes = ...  # your answer



## <font color="#ff5f2e">The logistic loss</font>

The **logistic loss** uses the natural log:
$$\text{Loss}_{\text{logistic}}(m) = \log\!\left(1 + e^{-m}\right).$$
At margin 0 it equals $\log 2 \approx 0.69$. For large negative margins it runs almost parallel to the hinge, with slope close to $-1$. For large positive margins it creeps toward 0 and never reaches it: about 0.31 at margin 1, 0.13 at 2, under 0.05 at 3.

Its slope never jumps. Using the chain rule, the slope with respect to the margin is $-\sigma(-m)$, where
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
is the **sigmoid**, the curve from the lesson on gradients that rises from 0 to 1. The slope is $-0.5$ at margin 0 and about $-0.12$ at margin 2. Confident, correct tickets barely move the weights.

The sigmoid also turns a score into a probability: $\sigma(\text{score})$ is the chance the ticket is urgent. A score of 0 gives one half. The two fit together exactly: the logistic loss is minus the log of the probability the model gives to the true label. Training with it makes the true labels as probable as possible, and the method is called **logistic regression**, even though it classifies.

The squared loss from the last lesson, with labels $\pm 1$, is $(\text{score} - y)^2 = (1 - m)^2$ on the margin axis: a parabola with its bottom at 1. It rises to the right as well, so a correct, confident ticket at margin 3 pays 4. It punishes being too right, and easy tickets drag the weights away from the hard ones.

That leaves two good choices. The hinge, when only the decision matters. The logistic loss, when you want a probability.

One numerical trap: `math.exp(-m)` overflows for a margin like $-800$. The library function `np.logaddexp(0, -m)`, which computes $\log(e^0 + e^{-m})$, gives the same value without the overflow.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
draw_losses(axes[0])
axes[0].set_title("Four answers to one question: what should this margin cost?")
z = np.linspace(-6, 6, 300)
axes[1].plot(z, 1 / (1 + np.exp(-z)), color=PALETTE["cyan"], lw=2)
for (text, _), s, colour in zip(TICKETS3, [1.0, -1.7, -0.1], [PALETTE["amber"], PALETTE["violet"], PALETTE["pink"]]):
    axes[1].scatter([s], [_sigmoid(s)], color=colour, s=70, zorder=3, label=text)
axes[1].axhline(0.5, color=PALETTE["ink"], ls=":", lw=0.8)
axes[1].set_xlabel("score"); axes[1].set_ylabel("probability of urgent"); axes[1].legend(fontsize=7)
axes[1].set_title("The sigmoid reads a score as a probability")
plt.tight_layout()
plt.show()



<font size="4"><b>The logistic loss</b></font>

Write `logistic(m)` with `math.log` and `math.exp`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The formula is in the notes above. `math.log` and `math.exp` cover it.

</details>

In [ ]:
# Exercise 4.1: running this cell checks your answer
def logistic(m: float) -> float:
    return ...  # your code here



<font size="4"><b>Values along the axis</b></font>

Make `log_vals`, the logistic loss at margins $-1$, 0, 1, 2 and 3.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

A list comprehension with your function.

</details>

In [ ]:
# Exercise 4.2: running this cell checks your answer
log_vals = ...  # your answer



<font size="4"><b>The sigmoid</b></font>

Write `sigmoid(z)`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The sigmoid formula is in the notes above. `math.exp` is all you need.

</details>

In [ ]:
# Exercise 4.3: running this cell checks your answer
def sigmoid(z: float) -> float:
    return ...  # your code here



<font size="4"><b>Three probabilities</b></font>

Make `p3`, the probability of urgent for each of the three tickets under `W1`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The sigmoid of each score. Your `score` and `PHI3` give the scores.

</details>

In [ ]:
# Exercise 4.4: running this cell checks your answer
p3 = ...  # your answer



<font size="4"><b>Loss from probability</b></font>

For the refund ticket (urgent), give `loss_from_p`, minus the log of the probability of its true label, and `loss_direct`, the logistic loss of its margin. Then for the routine ticket give `p_routine_true`, the probability of its true label.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The probability of routine is 1 minus the probability of urgent.

</details>

In [ ]:
# Exercise 4.5: running this cell checks your answer
loss_from_p = ...  # your answer
loss_direct = ...  # your answer
p_routine_true = ...  # your answer



<font size="4"><b>The slope</b></font>

Write `logistic_slope(m)`: the slope of the logistic loss with respect to the margin, $-\sigma(-m)$. Use your `sigmoid`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

One line.

</details>

In [ ]:
# Exercise 4.6: running this cell checks your answer
def logistic_slope(m: float) -> float:
    return ...  # your code here



<font size="4"><b>The overflow trap</b></font>

Try `logistic(-800)` in a scratch cell and you get an `OverflowError`. Write `logistic_safe(m)` with `np.logaddexp`, returning a plain float.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

`np.logaddexp(a, b)` computes $\log(e^a + e^b)$. Write the 1 inside the log as an exponential.

</details>

In [ ]:
# Exercise 4.7: running this cell checks your answer
def logistic_safe(m: float) -> float:
    return ...  # your code here



<font size="4"><b>The squared loss on margins</b></font>

Write `squared_m(m)` $= (1 - m)^2$. Then check it against the regression form: for label $-1$ and score $0.5$, store `sq_regression` $= (\text{score} - y)^2$ and `sq_margin`, your `squared_m` of the margin.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The margin is $-1 \times 0.5$.

</details>

In [ ]:
# Exercise 4.8: running this cell checks your answer
def squared_m(m: float) -> float:
    return ...  # your code here


sq_regression = ...  # your answer
sq_margin = ...  # your answer



<font size="4"><b>Too right</b></font>

A ticket has margin 3: correct and confident. Make the dictionary `cost_at_3` with keys `"zero-one"`, `"hinge"`, `"logistic"` and `"squared"`, using your functions.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Four calls.

</details>

In [ ]:
# Exercise 4.9: running this cell checks your answer
cost_at_3 = ...  # your answer



<font size="4"><b>The logistic valley</b></font>

Make `log_curve`, the average logistic loss of the three tickets at every weight in `WQ_GRID`, and `wq_log_best`, the grid weight where it is lowest. The plot appears when the check passes.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The same pattern as `hinge_at`, with `logistic`. Then `np.argmin`.

</details>

In [ ]:
# Exercise 4.10: running this cell checks your answer
log_curve = ...  # your answer
wq_log_best = ...  # your answer



<font size="4"><b>The library version</b></font>

PyTorch computes the mean logistic loss from scores and 0/1 targets with `torch.nn.functional.binary_cross_entropy_with_logits`. Turn the labels into targets with $(y + 1)/2$, compute the loss for the three tickets as `bce3`, a plain float, and compare it with the mean of your `logistic` over `margins3`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Make 64-bit tensors with `torch.tensor(..., dtype=torch.float64)` for the scores and the targets. `.item()` gives a float.

</details>

In [ ]:
# Exercise 4.11: running this cell checks your answer
scores3_t = ...  # your answer
targets3_t = ...  # your answer
bce3 = ...  # your answer



<font size="4"><b>A probability threshold</b></font>

The help desk wants every ticket with a probability of urgent above 0.9 sent straight to the engineer on call. Find `score_09`, the score where the sigmoid equals 0.9.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Solve $1 / (1 + e^{-s}) = 0.9$ for $s$: get $e^{-s}$ on its own first.

</details>

In [ ]:
# Exercise 4.12: running this cell checks your answer
score_09 = ...  # your answer



## <font color="#d0309f">The update</font>

Take the hinge loss of one ticket and find its gradient with respect to the weights. The margin is $y\,(w \cdot \phi)$, so a small change to the weight $w_j$ changes the margin by $y\,\phi_j$. Put that through the hinge with the chain rule:
$$\nabla_w \text{Loss}_{\text{hinge}} = \begin{cases} -y\,\phi(x) & m < 1 \\ 0 & m \ge 1. \end{cases}$$

One step of stochastic gradient descent subtracts the step size $\eta$ times the gradient:
$$\text{if } m < 1: \quad w \leftarrow w + \eta\, y\, \phi(x).$$
Otherwise the weights stay as they are.

In words: when an urgent ticket scores too low, every word in it becomes a little more urgent. When a routine ticket scores too high, every word in it becomes a little more routine. Words missing from the ticket don't change.

By hand, with step size 0.5 and all weights at 0: "app crashed again" (urgent) has margin 0, so each of its words gains 0.5. Then "app question thanks" (routine) scores 0.5 through the shared word "app", a margin of $-0.5$, so each of its words loses 0.5. "App" ends at 0: it appeared once on each side.

One update moves a ticket's own margin by $\eta \sum_j \phi_j^2$. Other tickets that share words move too.

A **pass** (or epoch) is one sweep through all the training tickets. On the 13 training tickets with step size 0.2, every margin starts at 0. After 3 passes every ticket has a margin of at least 1, the hinge loss is 0, and the weights stop changing.

The logistic version always updates, with a push of size $\sigma(-m)$: $w \leftarrow w + \eta\, \sigma(-m)\, y\, \phi(x)$.

In [ ]:
TRAIN = [
    ("login not working", 1), ("payment not going through", 1), ("app keeps crashing", 1),
    ("site is down", 1), ("card charged twice", 1), ("not urgent just a question", -1),
    ("everything is working now thanks", -1), ("printer working again thanks", -1),
    ("please do not close my account", -1), ("how do i change my email", -1),
    ("wifi not working at all", 1), ("upload is working now", -1), ("not a problem just asking", -1),
]
TRAIN_PHI = [(_phi_uni(t), y) for t, y in TRAIN]
UPD_PHI = [({"app": 1, "crashed": 1, "again": 1}, 1), ({"app": 1, "question": 1, "thanks": 1}, -1)]

w = {}
for phi, y in UPD_PHI + UPD_PHI:
    m = y * _dot(w, phi)
    w = _upd(w, phi, y, 0.5)
    print(f"margin {m:+.1f}  {'update' if m < 1 else 'no update':9}  weights {w}")

fig, ax = plt.subplots(figsize=(6.4, 3))
hist = _train_data(TRAIN_PHI, 0.2, 6)[1]
ax.plot(range(7), [13] + [h[0] for h in hist], "-o", color=PALETTE["pink"], label="mistakes")
ax2 = ax.twinx()
ax2.plot(range(7), [1.0] + [h[1] for h in hist], "-s", color=PALETTE["amber"], label="average hinge")
ax2.grid(False)
ax.set_xlabel("passes"); ax.set_ylabel("mistakes", color=PALETTE["pink"]); ax2.set_ylabel("average hinge", color=PALETTE["amber"])
ax.set_title("13 tickets, step size 0.2")
plt.show()



<font size="4"><b>The gradient</b></font>

Write `hinge_grad(w, phi, y)`: the gradient of one ticket's hinge loss as a dictionary. When the margin is below 1 it maps each feature to $-y\,\phi_j$; otherwise it is empty.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Compute the margin with your `score` and `margin`, then a dictionary comprehension over `phi.items()`.

</details>

In [ ]:
# Exercise 5.1: running this cell checks your answer
def hinge_grad(w: dict[str, float], phi: dict[str, float], y: int) -> dict[str, float]:
    ...  # your code here



<font size="4"><b>The update</b></font>

Write `hinge_update(w, phi, y, eta)` that returns a new weight dictionary after one hinge step. Leave the input `w` unchanged.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Copy `w` so the original stays unchanged. Only a margin below 1 changes anything, and each feature's step comes from the hinge gradient. Use `.get` with a default for words that have no weight yet.

</details>

In [ ]:
# Exercise 5.2: running this cell checks your answer
def hinge_update(w: dict[str, float], phi: dict[str, float], y: int, eta: float) -> dict[str, float]:
    ...  # your code here



<font size="4"><b>The first update by hand</b></font>

Start from empty weights and apply one update with step size 0.5 for the first ticket in `UPD_PHI`. Store the weights as `w_a` and the ticket's new margin as `m_a`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

`phi, y = UPD_PHI[0]`, then your `hinge_update` and `margin`.

</details>

In [ ]:
# Exercise 5.3: running this cell checks your answer
phi_a, y_a = UPD_PHI[0]
w_a = ...  # your answer
m_a = ...  # your answer



<font size="4"><b>The second update</b></font>

Now the second ticket of `UPD_PHI`, starting from `w_a`. Give its margin before the update, `m_b_before`, and the weights after, `w_b`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

It shares the word "app" with the first ticket.

</details>

In [ ]:
# Exercise 5.4: running this cell checks your answer
phi_b, y_b = UPD_PHI[1]
m_b_before = ...  # your answer
w_b = ...  # your answer



<font size="4"><b>The second pass</b></font>

Make `margins_pass2`, the margins of both tickets under `w_b`, and `any_update`, whether a second pass would change any weight.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

An update happens only for a margin below 1.

</details>

In [ ]:
# Exercise 5.5: running this cell checks your answer
margins_pass2 = ...  # your answer
any_update = ...  # your answer



<font size="4"><b>How far one update moves</b></font>

Write `margin_gain(phi, eta)`: how much one update moves the ticket's own margin, $\eta \sum_j \phi_j^2$.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Work out how much $y\,(w \cdot \phi)$ grows after one update, using $y^2 = 1$.

</details>

In [ ]:
# Exercise 5.6: running this cell checks your answer
def margin_gain(phi: dict[str, float], eta: float) -> float:
    return ...  # your code here



<font size="4"><b>Fix the near miss</b></font>

Apply one update with step size 0.5 to the refund ticket, `PHI3[2]`, starting from `W1`. Give the new weights `w_fix` and the ticket's new margin `m_fix`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Your `hinge_update`, then its margin.

</details>

In [ ]:
# Exercise 5.7: running this cell checks your answer
w_fix = ...  # your answer
m_fix = ...  # your answer



<font size="4"><b>The side effect</b></font>

"Question" also appears in the routine ticket "quick question thanks". Give its margin under `w_fix` as `m_side`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

The features are `PHI3[1]` and the label is $-1$.

</details>

In [ ]:
# Exercise 5.8: running this cell checks your answer
m_side = ...  # your answer



<font size="4"><b>Training</b></font>

Write `train_hinge(data, eta, epochs)`, where `data` is a list of `(phi, y)` pairs. Start from empty weights and apply your `hinge_update` to each ticket in order, for `epochs` passes. After each pass, record a pair: the number of mistakes and the average hinge loss over `data`. Return `(w, history)`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Two loops, passes outside and tickets inside. After the inner loop, compute every margin once and use your `zero_one` and `hinge`.

</details>

In [ ]:
# Exercise 5.9: running this cell checks your answer
def train_hinge(data: list[tuple[dict, int]], eta: float, epochs: int) -> tuple[dict, list[tuple[int, float]]]:
    ...  # your code here



<font size="4"><b>The learning curve</b></font>

Train on `TRAIN_PHI` with step size 0.2 for 5 passes and store the history as `history13`. The curve appears when the check passes.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

The second item that `train_hinge` returns.

</details>

In [ ]:
# Exercise 5.10: running this cell checks your answer
history13 = ...  # your answer



<font size="4"><b>The logistic update</b></font>

Write `logistic_update(w, phi, y, eta)`: always add $\eta\,\sigma(-m)\,y\,\phi_j$ to each feature's weight. Return a new dictionary. Then apply it to the refund ticket from `W1` with step size 0.5, as `w_log_fix`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

It has the shape of `hinge_update`. The logistic gradient from earlier tells you when a step happens and how big it is.

</details>

In [ ]:
# Exercise 5.11: running this cell checks your answer
def logistic_update(w: dict[str, float], phi: dict[str, float], y: int, eta: float) -> dict[str, float]:
    ...  # your code here


w_log_fix = ...  # your answer



<font size="4"><b>Already safe</b></font>

The ticket `{"crashed": 1, "again": 1}` is urgent and scores 1.7 under `W1`. Store `hinge_changes`, whether a hinge update would change the weights, and `push_17`, the size of the logistic push $\sigma(-m)$.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Check the margin against 1 first.

</details>

In [ ]:
# Exercise 5.12: running this cell checks your answer
hinge_changes = ...  # your answer
push_17 = ...  # your answer



## <font color="#0077ff">Word features</font>

A ticket arrives as text, and a linear classifier needs a list of numbers. The step between the two is ours to design.

A **feature template** is a rule that creates a whole family of features at once. The one we have used says: for each word, count how many times it appears. One rule gives thousands of features, and the training text decides which ones.

The **vocabulary** is the list of words that get a feature, usually every word in the training tickets, in a fixed order. Each vocabulary word is one entry of the feature vector, and every ticket, old or new, uses the same entries.

Before counting, the text is cleaned. Capitals are lowered so that "App" and "app" share one feature, and punctuation is stripped from the ends of words. Splitting the cleaned text into words is called **tokenizing**, and each word is a **token**.

Over the vocabulary `refund, not, received, card, blocked, please`, the ticket "please please refund my card" gives counts 1, 0, 0, 1, 0, 2. "My" is outside the vocabulary and is dropped.

This representation is a **bag of words**: it keeps which words appear and how often, and it throws away their order. A common variant records only presence: 1 if the word appears at all.

The 13 training tickets contain 39 different words. After training, "login", "site" and "down" lean hardest toward urgent, and "thanks", "now" and "my" are among the strongest pulls toward routine. Some of those leanings are accidents of a small data set.

In [ ]:
VOCAB6 = ["refund", "not", "received", "card", "blocked", "please"]

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2), gridspec_kw={"width_ratios": [1, 1.2]})
counts = [_phi_uni("please please refund my card").get(v, 0) for v in VOCAB6]
axes[0].imshow(np.array(counts)[None, :], cmap="magma", vmin=-0.5, vmax=2.5, aspect="auto")
for j, c in enumerate(counts):
    axes[0].text(j, 0, str(c), ha="center", va="center", color=PALETTE["night"] if c else PALETTE["ink"], weight="bold")
axes[0].set_xticks(range(6), VOCAB6, rotation=30); axes[0].set_yticks([]); axes[0].grid(False)
axes[0].set_title('"please please refund my card"')
draw_weights(axes[1], _W_UNI, "Learned word weights: the ends of the list", top=6)
plt.tight_layout()
plt.show()



<font size="4"><b>Tokenize</b></font>

Write `tokenize(text)`: lower the text, split it at whitespace, strip the characters in `PUNCT` from both ends of each word, and drop any word that becomes empty.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Lower-case and split the text. `str.strip` removes punctuation from both ends of a word; drop any word that ends up empty.

</details>

In [ ]:
# Exercise 6.1: running this cell checks your answer
PUNCT = ".,!?;:\"'()"


def tokenize(text: str) -> list[str]:
    return ...  # your code here



<font size="4"><b>Word counts</b></font>

Write `word_counts(text)`: a dictionary from each token to the number of times it appears. Use your `tokenize`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Start from an empty dictionary. `.get` with a default of 0 lets you add 1 to a count that may not exist yet.

</details>

In [ ]:
# Exercise 6.2: running this cell checks your answer
def word_counts(text: str) -> dict[str, int]:
    ...  # your code here



<font size="4"><b>A vector over the vocabulary</b></font>

Write `count_vector(text, vocab)`: a NumPy array of counts, one per vocabulary word, in the vocabulary's order. Words outside the vocabulary are dropped.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Count the words once with your `word_counts`, then read one entry per vocabulary word, with 0 for a missing word.

</details>

In [ ]:
# Exercise 6.3: running this cell checks your answer
def count_vector(text: str, vocab: list[str]) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>Order is lost</b></font>

Of the four tickets in `FOUR`, which two have identical bags of words? Store their positions as a sorted tuple `same_bag`, and find them with code.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Compare `word_counts` of every pair. `itertools` isn't needed: two nested loops over `range(4)` work.

</details>

In [ ]:
# Exercise 6.4: running this cell checks your answer
FOUR = ["login not working", "working login now", "not login working", "login working not now please"]
same_bag = ...  # your answer



<font size="4"><b>Presence only</b></font>

Write `presence(text)`: a dictionary with value 1 for every word that appears at least once.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

A dictionary comprehension over your `tokenize`.

</details>

In [ ]:
# Exercise 6.5: running this cell checks your answer
def presence(text: str) -> dict[str, int]:
    return ...  # your code here



<font size="4"><b>The training vocabulary</b></font>

Make `vocab13`, the sorted list of every distinct word in the texts of `TRAIN`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Collect tokens in a set, then `sorted(...)`.

</details>

In [ ]:
# Exercise 6.6: running this cell checks your answer
vocab13 = ...  # your answer



<font size="4"><b>Unknown words</b></font>

The new ticket "My laptop is on fire!" arrives. Count `dropped`, the number of its tokens that are outside `vocab13`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Tokenize it and count the tokens not in the list.

</details>

In [ ]:
# Exercise 6.7: running this cell checks your answer
dropped = ...  # your answer



<font size="4"><b>Train on word counts</b></font>

Build `word_data`, a list of `(word_counts(text), y)` for every ticket in `TRAIN`, and train with your `train_hinge`, step size 0.2, 20 passes. Store the weights as `w_words` and the set of the 3 words with the largest weights as `top3`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Build word-count data in the same form as `TRAIN_PHI` and train with your `train_hinge`. `sorted` with `key=` and `reverse=True` orders the words by weight.

</details>

In [ ]:
# Exercise 6.8: running this cell checks your answer
word_data = ...  # your answer
w_words = ...  # your answer
top3 = ...  # your answer



<font size="4"><b>Other templates</b></font>

Words are one template. Write `other_template(text)` that returns a dictionary with:

* `"length"`: the number of tokens;
* `"first=<word>"`: 1, where `<word>` is the first token;
* `"has_digit"`: 1, only if any character of the text is a digit;
* `"question_mark"`: 1, only if the text ends with `?` after stripping spaces.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Build the dictionary with the first two entries, then add the other two inside `if` statements. `ch.isdigit()` tests one character.

</details>

In [ ]:
# Exercise 6.9: running this cell checks your answer
def other_template(text: str) -> dict[str, int]:
    ...  # your code here



<font size="4"><b>The two tickets as bags</b></font>

Make `diff_words`, the set of words whose counts differ between the two tickets in `COLD`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Take the union of both key sets and keep the words where `.get(k, 0)` differs.

</details>

In [ ]:
# Exercise 6.10: running this cell checks your answer
bag_a, bag_b = ..., ...  # your answer
diff_words = ...  # your answer



<font size="4"><b>A count matrix</b></font>

Stack the training tickets into `X13`, a NumPy array of shape `(13, 39)`: one row per ticket from your `count_vector` over `vocab13`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

One row per ticket from your `count_vector`, stacked into an array.

</details>

In [ ]:
# Exercise 6.11: running this cell checks your answer
X13 = ...  # your answer



## <font color="#7b2ff7">Bigrams</font>

A bag of words can't see that "not" sits right before "working". A **bigram** is a pair of neighbouring words, and the bigram template counts how many times each pair appears.

Slide a window two words wide along the ticket. "the app is not working" gives 4 pairs: "the app", "app is", "is not" and "not working". A ticket of $n$ words has $n - 1$ bigrams, one per gap.

"the app is working" gives "the app", "app is" and "is working". The two tickets now differ in 4 features: "not", "is not" and "not working" on one side, and "is working" on the other.

We keep the single words too. The full feature dictionary holds the word counts and the bigram counts, and the classifier has one weight per entry. Trained on the 13 tickets with both templates, "not working" gets the largest weight of any feature, 0.6, and "is working" gets $-0.4$.

The single word "not" appears in 3 urgent tickets and 3 routine ones. The pair "not working" appears in 2 urgent tickets and no routine ones.

The price is size. The 13 tickets add 40 bigrams to the 39 words. With 50,000 words there are 2.5 billion possible pairs, though only pairs that occur in training get a feature. Longer windows work the same way: 3 words make a trigram, and the general name is **n-gram**. A longer phrase turns up in fewer tickets, so rare features learn from very little evidence.

In [ ]:
text = "the app is not working"
toks = text.split()
for k in range(len(toks) - 1):
    print(" ".join(f"[{t}]" if k <= j <= k + 1 else f" {t} " for j, t in enumerate(toks)))

bi_w = {k: v for k, v in _W_BI.items() if " " in k}
fig, ax = plt.subplots(figsize=(6.6, 4))
draw_weights(ax, bi_w, "Bigram weights after training: top and bottom", top=5)
plt.show()



<font size="4"><b>Bigrams</b></font>

Write `bigrams(tokens)`: the list of neighbouring pairs, each joined with one space.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`zip(tokens, tokens[1:])` lines up each word with the next.

</details>

In [ ]:
# Exercise 7.1: running this cell checks your answer
def bigrams(tokens: list[str]) -> list[str]:
    return ...  # your code here



<font size="4"><b>One per gap</b></font>

Make `n_gaps`, the number of bigrams of the 9-word ticket in `LONG`, using your functions.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Tokenize, then count the bigrams.

</details>

In [ ]:
# Exercise 7.2: running this cell checks your answer
LONG = "my order arrived broken and the box was wet"
n_gaps = ...  # your answer



<font size="4"><b>Words and pairs</b></font>

Write `features(text)`: the word counts plus the bigram counts, in one dictionary.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Start from `word_counts(text)` and add each bigram of `tokenize(text)` the same way.

</details>

In [ ]:
# Exercise 7.3: running this cell checks your answer
def features(text: str) -> dict[str, int]:
    ...  # your code here



<font size="4"><b>What separates the tickets now</b></font>

Make `diff_feats`, the set of features whose counts differ between the two tickets in `COLD`, with your `features`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The same pattern as `diff_words`.

</details>

In [ ]:
# Exercise 7.4: running this cell checks your answer
fa, fb = ..., ...  # your answer
diff_feats = ...  # your answer



<font size="4"><b>Order matters now</b></font>

Store `order_seen`: whether `features("working not")` and `features("not working")` differ.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Compare the two dictionaries.

</details>

In [ ]:
# Exercise 7.5: running this cell checks your answer
order_seen = ...  # your answer



<font size="4"><b>n-grams</b></font>

Write `ngrams(tokens, n)`: every run of `n` neighbouring tokens, joined with spaces.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Take each run of `n` neighbouring tokens with a slice and join it with spaces. Work out where the last run starts.

</details>

In [ ]:
# Exercise 7.6: running this cell checks your answer
def ngrams(tokens: list[str], n: int) -> list[str]:
    return ...  # your code here



<font size="4"><b>Which word, which pair</b></font>

Count in `TRAIN` how many urgent and how many routine tickets contain "not", as `not_counts = (urgent, routine)`, and the same for the pair "not working", as `nw_counts`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`"not" in features(text)` tests one ticket. Count separately for `y == 1` and `y == -1`.

</details>

In [ ]:
# Exercise 7.7: running this cell checks your answer
not_counts = ...  # your answer
nw_counts = ...  # your answer



<font size="4"><b>How many features</b></font>

Make `n_bigrams`, the number of distinct bigrams in the training texts, and `n_rare`, the number of distinct features (words or pairs) that appear in exactly one training ticket.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

For the second, count in how many tickets each feature appears: a dictionary from feature to number of tickets.

</details>

In [ ]:
# Exercise 7.8: running this cell checks your answer
n_bigrams = ...  # your answer
n_rare = ...  # your answer



<font size="4"><b>Possible n-grams</b></font>

Write `possible_ngrams(V, n)`: the number of possible n-grams from a vocabulary of `V` words. Then compute `pairs_50k` for 50,000 words.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Each of the `n` positions can hold any of the `V` words.

</details>

In [ ]:
# Exercise 7.9: running this cell checks your answer
def possible_ngrams(V: int, n: int) -> int:
    return ...  # your code here


pairs_50k = ...  # your answer



<font size="4"><b>Train with both templates</b></font>

Train with your `train_hinge` on `(features(text), y)` for every ticket in `TRAIN`, step size 0.2, 20 passes. Store the weights as `w_both` and the feature with the largest weight as `top_feature`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The same lines as for `w_words`, with `features`. `max(w_both, key=w_both.get)` finds the heaviest feature.

</details>

In [ ]:
# Exercise 7.10: running this cell checks your answer
both_data = ...  # your answer
w_both = ...  # your answer
top_feature = ...  # your answer



## <font color="#e52e71">Sparse vectors</font>

Write out one ticket's feature vector over a vocabulary of 50,000 words and it has 50,000 entries. The ticket "refund not received yet please" uses 5 of them; the rest are 0. A vector that is almost all zeros is **sparse**.

Stored in full at 8 bytes an entry, that vector takes 400 kilobytes. A million tickets would take 400 gigabytes, nearly all of it zeros. So we store only the features that are present, as a dictionary from feature name to count. The order of the entries doesn't matter, because each one carries its name.

The dot product needs only the present features too, since a zero feature times any weight adds nothing. The score is a short loop over the ticket's dictionary: look up each weight, multiply by the count, add. A word with no weight returns 0 from the lookup.

The update is sparse as well: it touches only the weights of the words in the ticket. Scoring a ticket or training on it takes time in proportion to the ticket's length, however large the vocabulary grows. The weights are the one thing kept for every feature, in a single table shared by every ticket.

In [ ]:
V_BIG = 50_000
present = [3, 977, 12_408, 30_115, 48_002]
row = np.zeros(V_BIG); row[present] = 1

fig, axes = plt.subplots(2, 1, figsize=(9, 2.8), gridspec_kw={"height_ratios": [1, 1.4]})
axes[0].imshow(row[None, :], cmap="magma", aspect="auto", interpolation="nearest")
axes[0].set_yticks([]); axes[0].set_xticks([0, 25_000, 49_999]); axes[0].grid(False)
axes[0].set_title("One ticket over 50,000 words: 5 boxes lit")
axes[1].barh(["dense, 1M tickets", "sparse, 1M tickets"], [V_BIG * 8 * 1e6 / 1e9, 5 * 16 * 1e6 / 1e9], color=[PALETTE["pink"], PALETTE["mint"]])
axes[1].set_xscale("log"); axes[1].set_xlabel("gigabytes (log scale)")
plt.tight_layout()
plt.show()



<font size="4"><b>How sparse</b></font>

Give `share_pct`, the percentage of the 50,000 entries that are non-zero for a 5-word ticket.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

A percentage is the part over the whole, times 100.

</details>

In [ ]:
# Exercise 8.1: running this cell checks your answer
share_pct = ...  # your answer



<font size="4"><b>The dense cost</b></font>

Write `dense_gb(V, n_tickets)`: gigabytes to store `n_tickets` full vectors of `V` entries at 8 bytes each (1 gigabyte is $10^9$ bytes). Then compute `gb_million` for 50,000 words and a million tickets.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Each entry takes 8 bytes. Count the entries across all tickets, then convert bytes to gigabytes.

</details>

In [ ]:
# Exercise 8.2: running this cell checks your answer
def dense_gb(V: int, n_tickets: int) -> float:
    return ...  # your code here


gb_million = ...  # your answer



<font size="4"><b>The sparse cost</b></font>

Suppose each stored entry takes 16 bytes: 8 for its position and 8 for its value. Give `ratio`, how many times smaller a 5-entry sparse ticket is than the full 50,000-entry vector.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Full bytes divided by sparse bytes.

</details>

In [ ]:
# Exercise 8.3: running this cell checks your answer
ratio = ...  # your answer



<font size="4"><b>A sparse score</b></font>

The ticket has features refund, not and late, each once. Use your `score` with the weights `W8` to give `score8`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

A word without a weight adds nothing.

</details>

In [ ]:
# Exercise 8.4: running this cell checks your answer
W8 = {"refund": 0.9, "not": 0.1, "received": 0.4, "crashed": 1.2, "thanks": -0.7}
score8 = ...  # your answer



<font size="4"><b>Counting the work</b></font>

Write `sparse_cost(phi)`: the numbers of lookups, multiplications and additions that the sparse score of `phi` takes, as a tuple.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Count the work for one present feature, then for $k$ of them. Adding up $k$ numbers takes fewer additions than there are numbers.

</details>

In [ ]:
# Exercise 8.5: running this cell checks your answer
def sparse_cost(phi: dict[str, float]) -> tuple[int, int, int]:
    return ...  # your code here



<font size="4"><b>Dense to sparse</b></font>

Write `to_sparse(vec, vocab)`: a dictionary from vocabulary word to value, holding only the non-zero entries of the array `vec`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Walk the vocabulary and the vector together and keep only the non-zero entries, as plain floats.

</details>

In [ ]:
# Exercise 8.6: running this cell checks your answer
def to_sparse(vec: np.ndarray, vocab: list[str]) -> dict[str, float]:
    return ...  # your code here



<font size="4"><b>Sparse to dense</b></font>

Write `to_dense(phi, vocab)`: the full NumPy array over the vocabulary. Features outside the vocabulary are dropped.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

One `.get(v, 0)` per vocabulary word.

</details>

In [ ]:
# Exercise 8.7: running this cell checks your answer
def to_dense(phi: dict[str, float], vocab: list[str]) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>Only the lit entries</b></font>

NumPy can do the same trick on a full array. For the arrays `x_big` and `w_big` below, find `idx`, the positions of the non-zero entries of `x_big`, and `score_fast`, the dot product using only those positions.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`np.flatnonzero` gives the positions of the non-zero entries. Take the dot product over those positions only.

</details>

In [ ]:
# Exercise 8.8: running this cell checks your answer
x_big = np.zeros(50_000); x_big[[3, 977, 12_408, 30_115, 48_002]] = [1, 1, 2, 1, 1]
w_big = np.random.default_rng(5).normal(size=50_000)
idx = ...  # your answer
score_fast = ...  # your answer



<font size="4"><b>A sparse update touches little</b></font>

Apply one hinge update with step size 0.5 to the ticket "refund not received yet please", labelled routine, starting from `W8`. Store the new weights as `w8_new` and `n_changed`, the number of features whose weight changed or was added.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Compare `w8_new.get(k)` with `W8.get(k)` for every key of `w8_new`.

</details>

In [ ]:
# Exercise 8.9: running this cell checks your answer
w8_new = ...  # your answer
n_changed = ...  # your answer



<font size="4"><b>Find the bug</b></font>

This in-place update crashes with a `KeyError` the first time a ticket has a new word: `w[k] = w[k] + eta * y * v`. Fix that line so a missing word starts from 0.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Read the old weight with `.get`.

</details>

In [ ]:
# Exercise 8.10: running this cell checks your answer
def add_update(w: dict[str, float], phi: dict[str, float], y: int, eta: float) -> dict[str, float]:
    """Adds eta * y * phi to w in place, and returns w."""
    for k, v in phi.items():
        w[k] = ...  # your answer
    return w



## <font color="#00897b">The classifier in code</font>

The whole classifier is now in your hands: `features` turns text into a sparse dictionary, `score` and `predict` make a decision, and `train_hinge` learns the weights. The weights start as an empty dictionary and get an entry the first time an update touches a feature.

To check a trained model, count the training tickets whose margin is 0 or below. That count only says the model fits the tickets it was trained on. How it does on new tickets is a separate question, which the next lesson takes up.

The same model also fits in a few lines of NumPy. Put the features of all tickets in a matrix $X$ of shape $(n, d)$, one row per ticket and one column per feature, and the labels in a vector $y$. Then the margins are $y \odot (Xw)$, where $\odot$ means entry by entry, and the subgradient of the average hinge loss is
$$-\frac{1}{n} X^\top \big(y \odot [m < 1]\big),$$
with $[m < 1]$ equal to 1 for tickets inside the safety gap and 0 otherwise.

In PyTorch you write the loss and let autograd find the gradient, as in the last lesson.

In [ ]:
FEATS = sorted({k for t, _ in TRAIN for k in _phi_bi(t)})
Y13 = np.array([y for _, y in TRAIN], dtype=float)
print(len(FEATS), "features, for example:", FEATS[:6])

fig, ax = plt.subplots(figsize=(10, 3))
ax.imshow(_X79, cmap="plasma", aspect="auto", interpolation="nearest")
ax.set_yticks(range(13), [f"{'+' if y > 0 else '−'} {t}" for t, y in TRAIN], fontsize=6)
ax.set_xlabel("feature column (words and pairs, sorted)"); ax.grid(False)
ax.set_title("13 tickets × 79 features: mostly zeros")
plt.show()



<font size="4"><b>The first update, both templates</b></font>

Start from empty weights. Apply one hinge update with step size 0.5 to "app crashed again", labelled urgent, using your `features`. Store the weights as `w_code`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

List the features first: three words and two pairs.

</details>

In [ ]:
# Exercise 9.1: running this cell checks your answer
w_code = ...  # your answer



<font size="4"><b>Training error</b></font>

Write `train_error(w, data)`: the number of `(phi, y)` pairs in `data` whose margin is 0 or below.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Count the mistakes with the functions you already have.

</details>

In [ ]:
# Exercise 9.2: running this cell checks your answer
def train_error(w: dict[str, float], data: list[tuple[dict, int]]) -> int:
    return ...  # your code here



<font size="4"><b>Errors pass by pass</b></font>

Train on `both_data` for 3 passes with step size 0.2 and store the mistake counts after each pass as `errs_both`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

The first entry of each pair in the history.

</details>

In [ ]:
# Exercise 9.3: running this cell checks your answer
errs_both = ...  # your answer



<font size="4"><b>Classify a ticket</b></font>

Write `classify(w, text)` that returns `"urgent"` or `"routine"`, using your `features`, `score` and `predict`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Turn the +1 or -1 from `predict` into a word.

</details>

In [ ]:
# Exercise 9.4: running this cell checks your answer
def classify(w: dict[str, float], text: str) -> str:
    return ...  # your code here



<font size="4"><b>New tickets</b></font>

Classify each ticket in `NEW4` with `w_both`, as a list `new_labels`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

A list comprehension with your `classify`.

</details>

In [ ]:
# Exercise 9.5: running this cell checks your answer
NEW4 = ["Checkout not working!", "thanks it is working now", "site keeps crashing", "my login is down"]
new_labels = ...  # your answer



<font size="4"><b>The feature matrix</b></font>

Build `X79`, the matrix with one row per ticket in `TRAIN` and one column per entry of `FEATS`, holding your `features` counts.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

One row per ticket and one column per feature in `FEATS`, with 0 for a feature the ticket lacks.

</details>

In [ ]:
# Exercise 9.6: running this cell checks your answer
X79 = ...  # your answer



<font size="4"><b>The subgradient as one line</b></font>

Write `hinge_grad_matrix(X, y, w)`: the subgradient of the average hinge loss, from the formula in the text.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Compute every margin at once with `X @ w`. A boolean mask of the margins below 1 picks the tickets that count, and `X.T @ ...` adds up their rows.

</details>

In [ ]:
# Exercise 9.7: running this cell checks your answer
def hinge_grad_matrix(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
    ...  # your code here



<font size="4"><b>Full-batch training in NumPy</b></font>

Start from `w_np = np.zeros(79)` and take 200 steps of size 0.5 with your `hinge_grad_matrix` on `X79` and `Y13`. Then store `margins_np = Y13 * (X79 @ w_np)`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Each step moves `w_np` against your `hinge_grad_matrix`, scaled by the step size.

</details>

In [ ]:
# Exercise 9.8: running this cell checks your answer
w_np = np.zeros(79)
for _ in range(200):
    w_np = ...  # your answer
margins_np = ...  # your answer



<font size="4"><b>The hinge with autograd</b></font>

Make a 64-bit tensor `w_t` of 79 zeros with `requires_grad=True`. Compute the average hinge loss `torch.clamp(1 - m, min=0).mean()` with `m = yt * (Xt @ w_t)`, call `.backward()`, and store `grad_torch = w_t.grad.numpy()`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`Xt = torch.tensor(X79)` and `yt = torch.tensor(Y13)` are already 64-bit.

</details>

In [ ]:
# Exercise 9.9: running this cell checks your answer
grad_torch = ...  # your answer



<font size="4"><b>Logistic regression in PyTorch</b></font>

Train logistic regression on `X79` and `Y13`: a 64-bit weight tensor of zeros, `torch.optim.SGD` with step size 0.5, and 300 steps of zero, forward, backward, step. The loss is the mean of `torch.nn.functional.softplus(-m)`, which equals the logistic loss. Store `p_torch`, the probabilities of urgent for the 13 tickets, as a NumPy array.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

The loop is the same as in the last lesson. At the end, `torch.sigmoid(Xt @ w).detach().numpy()`.

</details>

In [ ]:
# Exercise 9.10: running this cell checks your answer
p_torch = ...  # your answer



## <font color="#ff4d6d">What a line cannot do</font>

Shrink to two features, so each ticket is a point on a plane. The score is $w_1 x_1 + w_2 x_2 + b$, and the points where it equals 0 form a straight line, the **decision boundary**. Points on one side score positive and are called urgent; points on the other side are routine. The weights set the tilt of the line and the bias shifts it.

Data that one straight line can split this way are **linearly separable**. On separable data the hinge update keeps moving the line until every point clears the safety gap, and then stops. With 3 features the boundary is a flat plane, and with 50,000 it is a flat cut called a **hyperplane**.

Some patterns aren't separable. Put the urgent points in the middle and the routine points in a ring around them, and every line leaves routine points on both sides.

One fix is a new feature: the squared distance from the centre, $x_1^2 + x_2^2$. The middle points have values below 0.3 and the ring points above 2.5, so a single threshold on the new feature splits them. That threshold is a linear cut in the new features and a circle in the old picture. The fix needed us to guess the right feature; the next lesson builds models that learn their own.

In [ ]:
SEP_PTS = [((3, 2), 1), ((2.5, 3), 1), ((3.5, 3.2), 1), ((1, 0.8), -1), ((0.6, 1.8), -1), ((1.6, 0.4), -1)]
RING_IN = [(0.3, 0.2), (-0.4, 0.3), (0.1, -0.5), (-0.2, -0.2)]
RING_OUT = [(1.6, 0.2), (-1.5, 0.6), (0.4, 1.6), (-0.5, -1.5), (1.2, -1.1), (1.1, 1.2)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for (x1, x2), y in SEP_PTS:
    axes[0].scatter(x1, x2, s=80, color=PALETTE["amber"] if y > 0 else PALETTE["cyan"], zorder=3)
gx = np.linspace(0, 4, 2)
axes[0].plot(gx, 3 - gx, color=PALETTE["mint"], lw=2, label="x1 + x2 − 3 = 0")
axes[0].fill_between(gx, 3 - gx, 4, color=PALETTE["amber"], alpha=0.12)
axes[0].set_xlim(0, 4); axes[0].set_ylim(0, 4); axes[0].set_aspect("equal"); axes[0].legend(fontsize=8)
axes[0].set_title("Separable: one line splits them")
axes[1].scatter(*zip(*RING_IN), s=80, color=PALETTE["amber"], zorder=3, label="urgent")
axes[1].scatter(*zip(*RING_OUT), s=80, color=PALETTE["cyan"], zorder=3, label="routine")
gx = np.linspace(-2, 2, 2)
for slope, icpt in [(0.4, 0.6), (-1.5, -0.2), (3, 1)]:
    axes[1].plot(gx, slope * gx + icpt, color=PALETTE["pink"], lw=1.2, ls="--")
axes[1].set_xlim(-2, 2); axes[1].set_ylim(-2, 2); axes[1].set_aspect("equal"); axes[1].legend(fontsize=8)
axes[1].set_title("A ring: every line fails")
plt.tight_layout()
plt.show()



<font size="4"><b>Which side</b></font>

Write `side(w, b, x)` for NumPy arrays `w` and `x` and a number `b`: the prediction (+1 or -1) of the score $w \cdot x + b$. Use your `predict`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Compute the score with `@`, then reuse your `predict`.

</details>

In [ ]:
# Exercise 10.1: running this cell checks your answer
def side(w: np.ndarray, b: float, x: np.ndarray) -> int:
    return ...  # your code here



<font size="4"><b>The separable points</b></font>

With $w = (1, 1)$ and $b = -3$, make `sep_margins`, the margin of each point in `SEP_PTS`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Label times $(w \cdot x + b)$ for each point.

</details>

In [ ]:
# Exercise 10.2: running this cell checks your answer
sep_margins = ...  # your answer



<font size="4"><b>Draw the boundary</b></font>

Write `boundary_x2(w, b, x1)`: the second coordinate of the boundary point above `x1`, found by setting the score to 0. Assume $w_2 \ne 0$.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Solve $w_1 x_1 + w_2 x_2 + b = 0$ for $x_2$.

</details>

In [ ]:
# Exercise 10.3: running this cell checks your answer
def boundary_x2(w: np.ndarray, b: float, x1: float) -> float:
    return ...  # your code here



<font size="4"><b>Train the line</b></font>

Give each point the features $(x_1, x_2, 1)$, so the third weight is the bias. Run hinge updates with step size 0.1 in the order of `SEP_PTS`, pass after pass, until a whole pass makes no update. Store the weights as `w_sep` (a NumPy array of 3) and the number of passes, the quiet one included, as `passes_sep`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

A `while True` loop over passes with a flag that records whether any margin was below 1. With arrays, the update is `w = w + 0.1 * y * x`.

</details>

In [ ]:
# Exercise 10.4: running this cell checks your answer
w_sep, passes_sep = ..., ...  # your answer



<font size="4"><b>Every line fails on the ring</b></font>

Search lines $\cos\theta\, x_1 + \sin\theta\, x_2 + b$ over every angle in `ANGLES` and offset in `OFFSETS`, calling urgent the side with a positive score. Store `min_errors`, the fewest mistakes any of these lines makes on the 10 ring points.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Stack the points into one array `R` of shape `(10, 2)` and the labels into `L`. For each line, `np.where(R @ w + b > 0, 1, -1)` gives all 10 predictions at once.

</details>

In [ ]:
# Exercise 10.5: running this cell checks your answer
ANGLES = np.linspace(0, 2 * np.pi, 72, endpoint=False)
OFFSETS = np.linspace(-2, 2, 41)
min_errors = ...  # your answer



<font size="4"><b>A new feature</b></font>

Make the arrays `r2_in` and `r2_out`: the squared distance from the centre, $x_1^2 + x_2^2$, for the points in `RING_IN` and `RING_OUT`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

A list comprehension per group, wrapped in `np.array`.

</details>

In [ ]:
# Exercise 10.6: running this cell checks your answer
r2_in = ...  # your answer
r2_out = ...  # your answer



<font size="4"><b>One threshold, one circle</b></font>

Pick the threshold `t` halfway between the largest inner value and the smallest ring value. Store `ring_ok`, whether the rule "urgent if $x_1^2 + x_2^2 < t$" gets all 10 points right, and `radius`, the radius of the circle this rule draws in the original picture.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

The circle is where $x_1^2 + x_2^2 = t$.

</details>

In [ ]:
# Exercise 10.7: running this cell checks your answer
t = ...  # your answer
ring_ok = ...  # your answer
radius = ...  # your answer



<font size="4"><b>Three points on a line</b></font>

Urgent tickets sit at $(0, 0)$ and $(2, 2)$, a routine ticket at $(1, 1)$. For the line with $w = (0.3, -1.2)$ and $b = 0.5$, give `s_ends`, the scores at the two urgent points, and `s_mid`, the score at $(1, 1)$. Then store `separable`, whether any line can split these three points.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Compare `s_mid` with the average of `s_ends`. The score of a midpoint is always the average of the end scores.

</details>

In [ ]:
# Exercise 10.8: running this cell checks your answer
s_ends = ...  # your answer
s_mid = ...  # your answer
separable = ...  # your answer



<font size="4"><b>Lift one feature</b></font>

On a single feature, urgent tickets sit at $x = -2$ and $x = 2$, and a routine ticket at $x = 0$. Give each point the features $(x, x^2, 1)$ and train as in "Train the line": step size 0.1, in the order $-2, 0, 2$, until a quiet pass. Store the weights as `w_lift`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Reuse your loop with these three feature rows and the labels $(1, -1, 1)$.

</details>

In [ ]:
# Exercise 10.9: running this cell checks your answer
w_lift = ...  # your answer



## <font color="#1e88e5">The payoff</font>

Back to the two tickets: "the app is not working", which is urgent, and "the app is working", which is routine.

With word features alone, the trained weights give "app" and "not" 0.4 each, and "the", "is" and "working" 0. "Working" sits at 0 because it turned up on both sides. The routine ticket has nothing to pull it down, and both scores come out positive.

Add the bigram template and train again, with the same tickets, the same loss and the same step size. Only the features change. "Not working" lifts the first ticket and "is working" pulls the second one below 0.

The features decide which tickets the model can tell apart; the data decides how firmly. No template and no amount of training can separate two tickets whose features are identical.

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 2.6))
names = ["the", "app", "is", "not", "working", "the app", "app is", "is not", "not working", "is working"]
grid = np.array([[_phi_bi(t).get(k, 0) for k in names] for t, _ in COLD])
ax.imshow(grid, cmap="cool", aspect="auto", vmin=0, vmax=1.4)
ax.set_xticks(range(len(names)), names, rotation=35, fontsize=8); ax.set_yticks([0, 1], [t for t, _ in COLD]); ax.grid(False)
ax.set_title("Which features each ticket lights up, words and pairs")
plt.tight_layout()
plt.show()



<font size="4"><b>Scores with words alone</b></font>

Make `cold_uni`, the scores of the two tickets in `COLD` under `w_words`, using your `word_counts`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Your `score` for each text.

</details>

In [ ]:
# Exercise 11.1: running this cell checks your answer
cold_uni = ...  # your answer



<font size="4"><b>The whole gap</b></font>

Give `gap_uni`, the first score minus the second, and `w_not`, the weight of "not" in `w_words`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

The two bags differ only in "not".

</details>

In [ ]:
# Exercise 11.2: running this cell checks your answer
gap_uni = ...  # your answer
w_not = ...  # your answer



<font size="4"><b>Why &quot;working&quot; is zero</b></font>

Count how many urgent and routine training tickets contain "working", as `working_counts = (urgent, routine)`, and give its weight in `w_words` as `w_working`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

The same counting as `not_counts`.

</details>

In [ ]:
# Exercise 11.3: running this cell checks your answer
working_counts = ...  # your answer
w_working = ...  # your answer



<font size="4"><b>Scores with both templates</b></font>

Make `cold_bi`, the two scores under `w_both`, using your `features`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

The same as `cold_uni`.

</details>

In [ ]:
# Exercise 11.4: running this cell checks your answer
cold_bi = ...  # your answer



<font size="4"><b>What moved the second ticket</b></font>

Make `parts_b`, a dictionary from each feature of "the app is working" to its weight times count under `w_both`, and `pull_down`, the feature with the most negative part.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

A dictionary comprehension over `features(...)`, then `min(parts_b, key=parts_b.get)`.

</details>

In [ ]:
# Exercise 11.5: running this cell checks your answer
parts_b = ...  # your answer
pull_down = ...  # your answer



<font size="4"><b>The gap now</b></font>

Give `gap_bi`, the first score minus the second under `w_both`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

From `cold_bi`.

</details>

In [ ]:
# Exercise 11.6: running this cell checks your answer
gap_bi = ...  # your answer



<font size="4"><b>As probabilities</b></font>

Make `cold_p`, the probability of urgent for each ticket under `w_both`, with your `sigmoid`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

The sigmoid of each score in `cold_bi`.

</details>

In [ ]:
# Exercise 11.7: running this cell checks your answer
cold_p = ...  # your answer



<font size="4"><b>Inside the safety gap</b></font>

Make `cold_margins`, the two margins under `w_both`, and `inside_gap`, a list of booleans that are `True` where the margin is below 1.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Label times score for each ticket in `COLD`.

</details>

In [ ]:
# Exercise 11.8: running this cell checks your answer
cold_margins = ...  # your answer
inside_gap = ...  # your answer



<font size="4"><b>Identical features</b></font>

Take the urgent ticket "login not working" and a routine ticket "not login working". Store `words_differ`, whether their word counts differ, and `features_differ`, whether their word-and-pair features differ.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Compare `word_counts` of the two, then `features` of the two.

</details>

In [ ]:
# Exercise 11.9: running this cell checks your answer
words_differ = ...  # your answer
features_differ = ...  # your answer



<font size="4"><b>More data, firmer margins</b></font>

Add one more routine ticket, "checkout is working", to `TRAIN` and train again with both templates, step size 0.2, 20 passes. Store the new margin of "the app is working" (label routine) as `margin_more`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Add the new ticket to `TRAIN` and retrain the way you did for `w_both`. Then score the test ticket.

</details>

In [ ]:
# Exercise 11.10: running this cell checks your answer
w_more = ...  # your answer
margin_more = ...  # your answer



## <font color="#d84315">What you built</font>

* A linear classifier scores a ticket with a dot product and predicts its sign. The margin, label times score, is positive for a correct answer, and every loss here is a function of it.
* The zero-one loss counts mistakes. It is flat almost everywhere, so gradient descent can't use it, and we train on a surrogate loss instead.
* The hinge loss, $\max(0, 1 - m)$, is an upper bound on the mistakes with a safety gap at margin 1. The logistic loss, $\log(1 + e^{-m})$, is smooth, and the sigmoid of the score is a probability.
* The squared loss punishes tickets for being too right.
* One hinge step adds $\eta\, y\, \phi(x)$ to the weights when the margin is below 1, and leaves them alone otherwise.
* Feature templates build features from text: word counts give a bag of words, and bigrams keep a little of the order. Both are sparse, so dictionaries store them and the score loops only over the words present.
* A linear classifier draws one flat boundary. A new feature can make a curved pattern separable.

The next lesson stops hand-making features. The model learns them, and we ask how well it does on tickets it has never seen.



Run the cell below any time to see how far you are.

In [ ]:
progress()

<font size="2" color="#8a84b3">© 2026 Applied Data Science (<a href="https://www.youtube.com/@neuralmodules" style="color:#ff8fc4">youtube.com/@neuralmodules</a>). All rights reserved. This notebook and the lesson video belong to the channel. They are for personal, non-commercial study. You may run and change your own copy. You may not republish, share, sell or reproduce this notebook, the videos or any part of them without written permission from the channel.</font>